# 🫧 13 · Mixture models and the EM algorithm
### Soft clustering with probabilities — Gaussian and binomial mixtures, the EM algorithm and its guarantees, from cell states to population structure

**BiologyToML** · Level 1 · 02-620 core · Part C — Unsupervised learning · Module 13 of 44

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/13_mixture_models_em.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy (`scipy.stats`), pandas, Matplotlib, scikit-learn (for checking only) |
| **Prerequisites** | [Module 02](02_probability_for_biologists.ipynb) (probability, Bayes' rule, the Gaussian), [Module 05](05_likelihood_mle_map.ipynb) (likelihood, MLE) and [Module 12](12_clustering_kmeans_hierarchical.ipynb) (k-means); helpful: [Module 03](03_linear_algebra_for_data.ipynb) (vectors and matrices) |
| **Estimated time** | 180 min (reading + running + the ✏️ exercises) |
| **02-620 topics covered** | Mixture models: clustering as reordering samples (gene time series, genotypes of individuals), hard vs soft (probabilistic) assignment, the Gaussian mixture model as a two-step generative story and as a Bayesian network c → x (the bridge to hidden Markov models), inference of cluster labels by Bayes' rule, maximum-likelihood learning with observed labels, the EM algorithm for hidden labels (optimisation criterion, E- and M-steps, convergence), k-means as hard-assignment EM, EM in practice (initialisation, restarts), choosing the number of clusters (held-out likelihood, Dirichlet-process prior / Chinese restaurant process), Bernoulli mixtures for discrete data, comparing clustering methods and internal/external evaluation (GO enrichment) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> **All data are SIMULATED.** The leukaemia expression matrix (Sections 7.2 and 10) comes from the toolkit's
> `simulate_leukemia_expression`, shaped like Golub *et al.* (1999) — the values are not the paper's. The gene time series
> and genotype matrices of Section 1, the 1-D "marker gene" cells, the 2-D three-state and five-group point clouds, the
> genotypes of three populations (Section 9) and the Gene-Ontology numbers of Section 11 are simulated or invented toy
> numbers, labelled as such.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** why soft (probabilistic) cluster assignments are more informative than the hard assignments of k-means.
2. **Write down** the Gaussian mixture model as a two-step generative story and as a two-node Bayesian network c → x, and **count** its parameters.
3. **Compute** cluster responsibilities P(c = i | x) with Bayes' rule — by hand and, stably, in log space.
4. **Derive** the maximum-likelihood estimates when labels are observed, and the E- and M-steps of EM when they are not; **implement** EM from scratch for 1-D and 2-D Gaussian mixtures and for binomial (genotype) mixtures.
5. **State precisely** what EM guarantees — the observed-data log-likelihood never decreases — and what it does not: a global optimum, or even a finite optimum (degenerate solutions).
6. **Show** numerically that k-means is the hard-assignment limit of a Gaussian mixture with shared spherical covariance σ²I as σ → 0.
7. **Choose** the number of clusters with held-out likelihood and BIC, and **evaluate** clusterings with internal (silhouette) and external (ARI, GO enrichment) criteria.
8. **Interpret** mixture-model results on (simulated) leukaemia data, including subtype discovery, over-confident posteriors and artefacts of preprocessing.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit | 5 min |
| 1 · From hard to soft clustering | clustering reorders samples; k-means table vs probability table | 15 min |
| 2 · The Gaussian mixture model | generative story, multimodal densities, c → x | 20 min |
| 3 · Inference: who belongs where? | responsibilities by Bayes' rule, log space | 15 min |
| 4 · Learning when labels are observed | MLE by counting and averaging | 10 min |
| 5 · Labels hidden: the EM algorithm | E-step, M-step, the lower bound, 1-D and 2-D EM | 30 min |
| 6 · k-means as a limit of EM | σ → 0 turns probabilities into 0/1 | 10 min |
| 7 · EM in practice | restarts; degenerate solutions on expression data | 20 min |
| 8 · How many clusters? | held-out likelihood, BIC, the Chinese restaurant process | 15 min |
| 9 · Binomial mixtures and population structure | EM on SIMULATED genotypes of three populations | 20 min |
| 10 · A mixture model of leukaemia samples | diagonal-covariance GMM on the SIMULATED leukaemia data; subtypes | 15 min |
| 11 · Comparing and evaluating clusterings | comparison table, silhouette, ARI, GO enrichment | 10 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 5 min |

## 🧫 The biological question

Picture a genotype matrix — $N$ people × $J$ genetic loci, one row per person — whose rows can be re-ordered into
groups that line up with continental ancestry (African, Asian, European). Programs such as **STRUCTURE** (Pritchard,
Stephens & Donnelly 2000) and **ADMIXTURE** (Alexander, Novembre & Lange 2009) do exactly this for thousands of people
genotyped at hundreds of thousands of SNPs, and they are built on the model of this notebook: a **mixture model** in which each individual's genotypes
are drawn from the allele frequencies of a hidden population.

Why not just use k-means (Module 12)? Because real people, cells and genes do not always belong cleanly to one group.
An individual with grandparents from two continents, a T cell caught half-way through activation, a gene regulated by two
programmes — for all of them, the honest answer to "which cluster?" is a **probability**: *70 % this one, 30 % that one*.
Mixture models give such **soft assignments**, and they come with a **likelihood**, so we can compare models, choose the
number of clusters and reason about uncertainty with the probability tools of Modules 02 and 05.

The same machinery appears all over computational biology: assigning RNA-seq reads that map to several genes or isoforms
to the transcript they came from, calling genotypes from genotyping-array intensities, separating cell populations in flow
cytometry, and — in the form of hidden Markov models — finding genes and CpG islands in DNA
([Modules 19–20](19_hmm_inference.ipynb)). The algorithm that learns all of them is the **expectation–maximisation (EM)
algorithm** (Dempster, Laird & Rubin 1977).

> 🧪 **Simulated data, with a known truth.** The course stores no data sets: every data set in this notebook is generated
> by code. The leukaemia expression data of Sections 7 and 10 come from the toolkit's `simulate_leukemia_expression`,
> which imitates the classic study of Golub *et al.* (1999) — 72 patients (47 ALL, of them 38 B-lineage and 9 T-lineage,
> and 25 AML), 3,571 genes on a log10 scale, a detection floor at 2.0 and marker genes named after those that
> distinguished the leukaemias in the paper. Simulated data have a second advantage: we know the truth (which cluster
> every point came from, which subtype every patient has), so we can check whether a clustering found it. None of the
> numbers below are the paper's numbers.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import to_rgb
from matplotlib.patches import Ellipse
from scipy import stats
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.mixture import GaussianMixture

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
`simulate_leukemia_expression` generates the SIMULATED leukaemia expression data (shaped like Golub *et al.* 1999), `simulate_genotypes` draws Hardy–Weinberg genotypes for several populations, `logsumexp` computes log Σ exp stably, `draw_graph` draws small graphs and `CLASS_COLORS` is the course palette. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import os
import numpy as np
import matplotlib.pyplot as plt

LEUKEMIA_MARKERS = {
    "CST3": ("AML", "cystatin C"), "ZYX": ("AML", "zyxin"), "CD33": ("AML", "CD33 antigen"),
    "MPO": ("AML", "myeloperoxidase"), "LYZ": ("AML", "lysozyme"), "CTSD": ("AML", "cathepsin D"),
    "FAH": ("AML", "fumarylacetoacetate hydrolase"), "LTC4S": ("AML", "leukotriene C4 synthase"),
    "LYN": ("AML", "LYN kinase"), "HOXA9": ("AML", "homeobox A9"), "CFD": ("AML", "adipsin"),
    "LEPR": ("AML", "leptin receptor"), "SRGN": ("AML", "proteoglycan 1 (serglycin)"),
    "CXCL8": ("AML", "interleukin-8"), "AZU1": ("AML", "azurocidin"), "MCL1": ("AML", "MCL1"),
    "ITGAX": ("AML", "CD11c"), "NFKBIA": ("AML", "MAD-3 (IκBα)"), "ELANE": ("AML", "neutrophil elastase"),
    "CEBPA": ("AML", "C/EBPα"), "CSF3R": ("AML", "G-CSF receptor"), "FCER1G": ("AML", "Fc receptor γ"),
    "TYROBP": ("AML", "DAP12"), "S100A9": ("AML", "calprotectin subunit"), "CFP": ("AML", "properdin"),
    "MYB": ("ALL", "c-Myb"), "CCND3": ("ALL", "cyclin D3"), "RBBP4": ("ALL", "RbAp48"),
    "SMARCA4": ("ALL", "SNF2 (BRG1)"), "TCF3": ("ALL", "E2A"), "DUT": ("ALL", "dUTPase"),
    "STMN1": ("ALL", "Op18 (stathmin)"), "IL7R": ("ALL", "IL-7 receptor"), "RABEP1": ("ALL", "rabaptin-5"),
    "CBX3": ("ALL", "heterochromatin protein p25"), "DNTT": ("ALL", "terminal deoxynucleotidyl transferase"),
    "CD24": ("ALL", "CD24"), "TOP2B": ("ALL", "topoisomerase II β"), "IRF2": ("ALL", "IRF-2"),
    "MYL6": ("ALL", "myosin light chain"), "PSMB8": ("ALL", "proteasome subunit"), "LMO2": ("ALL", "LMO2"),
    "CD79A": ("B-ALL", "MB-1 (Igα)"), "CD19": ("B-ALL", "CD19"), "CD79B": ("B-ALL", "Igβ"),
    "VPREB1": ("B-ALL", "pre-B receptor"), "PAX5": ("B-ALL", "PAX5"), "CD22": ("B-ALL", "CD22"),
    "CD3E": ("T-ALL", "CD3ε"), "CD3D": ("T-ALL", "CD3δ"), "CD2": ("T-ALL", "CD2"), "CD7": ("T-ALL", "CD7"),
    "LCK": ("T-ALL", "LCK kinase"), "CD1E": ("T-ALL", "CD1e"),
}


def simulate_leukemia_expression(seed: int = 1999) -> dict:
    """Synthetic leukaemia expression data shaped like the classic Golub et al. (1999) study.

    72 bone-marrow/blood samples — 47 acute lymphoblastic leukaemia (ALL: 38 B-lineage, 9 T-lineage)
    and 25 acute myeloid leukaemia (AML) — × 3,571 genes on a log10 microarray scale, with the
    original 38/34 training/test design (train: 19 B-ALL, 8 T-ALL, 11 AML).  Values below the
    detection limit are floored at 2.0 (≈ 23 % of entries) and saturate at 4.2.  The 54 genes of
    ``LEUKEMIA_MARKERS`` carry the strongest signals (42 differ between ALL and AML by 0.5–1.0
    log10 units; 12 mark B- or T-lineage ALL), 900 more genes differ modestly between ALL and AML,
    150 separate T-ALL from B-ALL, and 15 hidden co-expression programmes correlate genes.  Two
    AML samples are atypical (signature at 35 % strength).  The classes are cleaner than in the
    real study, so many classifiers make few or no errors.

    Returns a dict with ``X`` (72 × 3,571), ``y`` (0 = ALL, 1 = AML), ``labels``, ``subtype``,
    ``genes``, ``is_train``, ``sample_id`` (1–72), ``atypical`` (True for the two weak-signature AML
    samples), ``true_effect`` and ``source``.  ``true_effect`` is the planted AML − ALL difference
    per gene; it is 0 for the 12 lineage markers and the 150 T-vs-B genes, although the B-lineage
    markers still differ between ALL and AML through the subtype shifts (by about 0.8 log10 units).
    The default ``seed`` gives the course's standard data set.
    """
    rng = np.random.default_rng(seed)
    J = 3571
    subtype = np.array(["B-ALL"] * 19 + ["T-ALL"] * 8 + ["AML"] * 11 + ["B-ALL"] * 19 + ["T-ALL"] * 1 + ["AML"] * 14,
                       dtype=object)
    is_train = np.r_[np.ones(38, bool), np.zeros(34, bool)]
    order = np.r_[rng.permutation(38), 38 + rng.permutation(34)]          # mix classes within each set
    subtype, is_train = subtype[order], is_train[order]
    n = len(subtype)
    y = (subtype == "AML").astype(int)
    named = list(LEUKEMIA_MARKERS)
    genes = np.array(named + [f"G{j:04d}" for j in range(1, J - len(named) + 1)], dtype=object)
    low = rng.random(J) < 0.4                                             # genes mostly below detection
    mu = np.where(low, rng.uniform(1.5, 2.4, J), np.clip(rng.normal(2.9, 0.4, J), 2.2, 4.0))
    sd = rng.uniform(0.12, 0.35, J)
    X = mu + sd * rng.normal(0, 1, (n, J))
    factors = rng.normal(0, 1, (n, 15))                                  # co-expression programmes
    loadings = rng.normal(0, 0.15, (15, J)) * (rng.random((15, J)) < 0.08)
    X += factors @ loadings
    effect = np.zeros(J)
    strong = np.arange(len(named))
    for j, g in enumerate(named):
        cls = LEUKEMIA_MARKERS[g][0]
        size = rng.uniform(0.5, 1.0)
        effect[j] = size if cls == "AML" else -size if cls == "ALL" else 0.0
        mu[j] = max(mu[j], 2.5)
    moderate = rng.choice(np.arange(len(named), J), 900, replace=False)
    effect[moderate] = rng.choice([-1, 1], 900) * rng.uniform(0.1, 0.4, 900)
    strength = np.ones(n)
    strength[np.flatnonzero(y == 1)[rng.choice((y == 1).sum(), 2, replace=False)]] = 0.35   # atypical AML
    X[:, strong] = mu[strong] + sd[strong] * rng.normal(0, 1, (n, len(strong)))
    X += np.outer((y - y.mean()) * strength, effect)
    for g, (cls, _) in LEUKEMIA_MARKERS.items():
        if cls in ("B-ALL", "T-ALL"):
            j = named.index(g)
            X[:, j] += np.where(subtype == cls, 0.9, np.where(subtype == "AML", -0.2, -0.5))
    lineage = rng.choice(np.setdiff1d(np.arange(len(named), J), moderate), 150, replace=False)
    X[:, lineage] += np.outer(np.where(subtype == "T-ALL", 1.0, np.where(subtype == "B-ALL", -0.2, 0.0)),
                              rng.choice([-1, 1], 150) * rng.uniform(0.4, 0.8, 150))
    X = np.round(np.clip(X, 2.0, np.log10(16000)), 4)
    return dict(X=X, y=y, labels=np.where(y == 1, "AML", "ALL").astype(object), subtype=subtype, genes=genes,
                is_train=is_train, sample_id=np.arange(1, n + 1), atypical=strength < 1, true_effect=effect,
                source="SIMULATED leukaemia data (shaped like Golub et al. 1999)")


def simulate_genotypes(rng: np.random.Generator, n_samples: int, n_snps: int, n_pops: int = 1,
                       fst: float = 0.0, maf_range=(0.05, 0.5)):
    """Simulate 0/1/2 genotypes under Hardy–Weinberg equilibrium.

    Each SNP gets an ancestral allele frequency drawn uniformly from ``maf_range``.  With
    ``n_pops > 1`` and ``fst > 0`` every population's frequency is drawn from the
    Balding–Nichols model Beta(p(1−F)/F, (1−p)(1−F)/F).  Samples are split evenly across
    populations.  Returns ``(G, pop, freqs)``: the N × J genotype matrix, the population of
    each sample and the n_pops × J matrix of population allele frequencies.

    ``G`` is stored as int8 to save memory: convert with ``G.astype(float)`` before arithmetic
    whose intermediate results can exceed 127.  A rare SNP can come out monomorphic (no
    variation) in a small sample — skip such columns in per-SNP tests.
    """
    p = rng.uniform(*maf_range, n_snps)
    if n_pops > 1 and fst > 0:
        a, b = p * (1 - fst) / fst, (1 - p) * (1 - fst) / fst
        freqs = np.clip(rng.beta(a, b, size=(n_pops, n_snps)), 0.001, 0.999)
    else:
        freqs = np.tile(p, (n_pops, 1))
    pop = np.repeat(np.arange(n_pops), int(np.ceil(n_samples / n_pops)))[:n_samples]
    G = rng.binomial(2, freqs[pop]).astype(np.int8)
    return G, pop, freqs


def logsumexp(a, axis=None):
    """log(Σ exp(a)) computed stably (subtract the maximum first)."""
    a = np.asarray(a, float)
    m = np.max(a, axis=axis, keepdims=True)
    m = np.where(np.isfinite(m), m, 0.0)
    out = np.log(np.sum(np.exp(a - m), axis=axis, keepdims=True)) + m
    return np.squeeze(out, axis=axis) if axis is not None else out.item()


CLASS_COLORS = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#ff7f0e", "#8c564b"]


def draw_graph(edges, pos: dict, directed: bool = True, ax=None, highlight=None, node_color: str = "#9ecae1",
               highlight_color: str = "#fdae6b", radius: float = 0.07, title: str = "", edge_labels=None):
    """Draw a small graph with Matplotlib only.

    ``edges`` is a list of (u, v) pairs, ``pos`` maps every node to (x, y) in [0, 1]².
    ``highlight`` is a set of nodes to colour differently (e.g. observed evidence).
    """
    from matplotlib.patches import Circle, FancyArrowPatch
    ax = ax or plt.gca()
    highlight = set(highlight or [])
    for i, (u, v) in enumerate(edges):
        (x0, y0), (x1, y1) = pos[u], pos[v]
        d = np.hypot(x1 - x0, y1 - y0) or 1.0
        sx, sy = x0 + (x1 - x0) * radius / d, y0 + (y1 - y0) * radius / d
        ex, ey = x1 - (x1 - x0) * radius / d, y1 - (y1 - y0) * radius / d
        ax.add_patch(FancyArrowPatch((sx, sy), (ex, ey), arrowstyle="-|>" if directed else "-",
                                     mutation_scale=12, lw=1.2, color="0.25"))
        if edge_labels is not None:
            ax.text((x0 + x1) / 2, (y0 + y1) / 2, str(edge_labels[i]), fontsize=7, color="0.2",
                    ha="center", va="center", bbox=dict(fc="white", ec="none", alpha=0.7, pad=0.5))
    for node, (x, y) in pos.items():
        ax.add_patch(Circle((x, y), radius, fc=highlight_color if node in highlight else node_color,
                            ec="0.2", lw=1, zorder=3))
        ax.text(x, y, str(node), ha="center", va="center", fontsize=9, zorder=4)
    ax.set_xlim(-0.1, 1.1); ax.set_ylim(-0.1, 1.1); ax.set_aspect("equal"); ax.axis("off")
    if title:
        ax.set_title(title)
    return ax


print("🧰 Toolkit loaded:", ", ".join(['simulate_leukemia_expression', 'simulate_genotypes', 'logsumexp', 'draw_graph']))

---
## 1 · From hard to soft clustering

> 📍 **Where we are.** In [Module 12](12_clustering_kmeans_hierarchical.ipynb) k-means split samples into $K$ groups:
> given samples $\mathbf x^1, \dots, \mathbf x^N$, each with $J$ variables $\mathbf x^n = [x^n_1, \dots, x^n_J]$, find $K$
> clusters. (Notation: in this module the sample index $n$ sits **up** as a superscript, so that subscripts can index
> variables and clusters.) Here we keep the goal but change the *kind* of answer.

**Clustering groups similar samples together.** Think of the data as a table with $N$ rows (samples) and $J$ columns
(variables). Clustering finds $K$ groups and **reorders the rows** so that members of the same cluster sit next to each
other — the hidden block structure becomes visible. Two classic biological examples:

- **Gene expression time series.** In yeast microarray time courses every gene gives one curve over time; clustering the
  curves groups genes with the same temporal programme (induced early, induced late, repressed…), and each cluster is
  summarised by its **centroid** — the average curve of its members.
- **Genotypes.** Genotype data collected for $N$ individuals at $J$ loci; clustering the individuals groups them into
  populations (for example African, Asian and European ancestry). Here the data are **discrete** (0, 1 or 2 copies of an
  allele), which will call for a different kind of mixture in Section 9.

The next cell simulates both kinds of data and clusters them with the k-means of Module 12.

In [ ]:
# ── 1.1 Clustering reorders samples: gene time courses and genotypes (both SIMULATED) ──
def kmeans_hard(X, K, init=None, n_starts=10, seed=0):
    """Compact k-means of Module 12 (best of n_starts random starts unless init is given). Returns labels, centres."""
    best = None
    for s in range(1 if init is not None else n_starts):
        rng_s = np.random.default_rng(seed + s)
        cen = np.array(init, float) if init is not None else X[rng_s.choice(len(X), K, replace=False)].copy()
        for _ in range(100):
            lab = ((X[:, None, :] - cen[None]) ** 2).sum(2).argmin(1)
            new = np.array([X[lab == k].mean(0) if np.any(lab == k) else cen[k] for k in range(K)])
            if np.allclose(new, cen):
                break
            cen = new
        obj = ((X - cen[lab]) ** 2).sum()
        if best is None or obj < best[2]:
            best = (lab, cen, obj)
    return best[0], best[1]


rng = np.random.default_rng(SEED + 10)
t_h = np.linspace(0, 12, 13)                                            # 13 time points, hours
programmes = np.array([np.tanh((t_h - 3) / 1.5), -np.tanh((t_h - 4) / 2), 1.6 * np.exp(-(t_h - 6) ** 2 / 4) - 0.5,
                       np.sin(2 * np.pi * t_h / 12)])                   # induced, repressed, transient, periodic
prog = rng.integers(0, 4, 120)                                          # 120 genes, one programme each
curves = programmes[prog] + rng.normal(0, 0.35, (120, len(t_h)))       # log fold change vs time 0
lab_ts, cen_ts = kmeans_hard(curves, 4)

G1, pop1, _ = simulate_genotypes(rng, n_samples=90, n_snps=60, n_pops=3, fst=0.3)
shuffle = rng.permutation(90)
G1, pop1 = G1[shuffle].astype(float), pop1[shuffle]                     # rows in random order, as collected
lab_g1, _ = kmeans_hard(G1, 3)
print("time-course clusters (genes per cluster):", np.bincount(lab_ts))
print("genotype clusters vs true populations:\n", pd.crosstab(lab_g1, pop1, rownames=["k-means cluster"],
                                                           colnames=["population"]))

fig, axes = plt.subplots(1, 4, figsize=(12, 3), gridspec_kw=dict(width_ratios=[1.2, 1.2, 1, 1]))
axes[0].plot(t_h, curves.T, color="0.75", lw=0.5)
axes[0].set_title("120 gene time courses\n(one curve per gene)", fontsize=9)
for k in range(4):
    axes[1].plot(t_h, cen_ts[k], lw=2.5, color=CLASS_COLORS[k], label=f"cluster {k + 1} ({np.sum(lab_ts == k)} genes)")
axes[1].set_title("k-means cluster centroids\n(average curve of each cluster)", fontsize=9); axes[1].legend(fontsize=6)
for ax in axes[:2]:
    ax.set_xlabel("time (h)")
axes[0].set_ylabel("log fold change")
axes[2].imshow(G1, aspect="auto", cmap="Greys", interpolation="nearest")
axes[2].set_title("genotypes, N = 90 × J = 60\n(rows as collected)", fontsize=9)
order1 = np.argsort(lab_g1, kind="stable")
axes[3].imshow(G1[order1], aspect="auto", cmap="Greys", interpolation="nearest")
for b in np.flatnonzero(np.diff(lab_g1[order1])) + 0.5:
    axes[3].axhline(b, color="tab:red", lw=1.5)
axes[3].set_title("rows reordered by cluster\n(red lines: cluster borders)", fontsize=9)
for ax in axes[2:]:
    ax.set_xlabel("locus j"); ax.set_yticks([])
axes[2].set_ylabel("individuals")
fig.suptitle("clustering groups similar samples (SIMULATED time courses and genotypes)", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Left: 120 noisy gene curves look like a tangle, but k-means finds four groups whose
centroids are clean temporal programmes — induced, repressed, a transient peak and a periodic wave (the colours and
order of the clusters are arbitrary). Right: in the genotype matrix as collected (rows in random order) no structure is
visible; after reordering the rows by cluster, blocks of loci where one group carries more dark (two-copy) genotypes
appear, and the three clusters coincide exactly with the three simulated populations (the table above). (The
populations were made strongly different, $F_{ST} = 0.3$, to make the blocks visible; Section 9 treats realistic ones.)

**Hard versus soft.** k-means gives a **hard assignment**: a table with one row per individual and a single 1 in the column
of its cluster. That hides an important distinction: a point right next to a cluster centre is *certain*, while a point
half-way between two centres is *uncertain* — yet both get a plain "1". **Probabilistic soft clustering** instead fills
the table with probabilities: each row says how likely the individual is to belong to each cluster, and every row sums to 1.

We continue with a SIMULATED data set of 150 samples measured in two conditions, generated from three groups with
different shapes. The function `sample_gmm` implements the generative story we explain in Section 2.

In [ ]:
# ── 1.2 Three groups of samples (SIMULATED) and their k-means hard assignment ──
PI_TRUE = np.array([0.3, 0.3, 0.4])                                    # mixing proportions
MU_TRUE = np.array([[3.9, 3.6], [1.6, 1.8], [3.3, 1.5]])               # cluster means (2 × 1 each)
SIGMA_TRUE = np.array([[[0.18, 0.10], [0.10, 0.18]],                   # compact, tilted
                       [[0.15, 0.00], [0.00, 0.50]],                   # tall and narrow
                       [[0.45, -0.25], [-0.25, 0.40]]])                # long diagonal


def sample_gmm(rng, N, pis, mus, Sigmas):
    """Draw N samples: first a cluster label c ~ Categorical(pis), then x | c ~ N(mus[c], Sigmas[c])."""
    c = rng.choice(len(pis), size=N, p=pis)
    X = np.array([rng.multivariate_normal(mus[k], Sigmas[k]) for k in c])
    return X, c


rng = np.random.default_rng(SEED + 1)
X, c_true = sample_gmm(rng, 150, PI_TRUE, MU_TRUE, SIGMA_TRUE)
lab_km, cen_km = kmeans_hard(X, 3)
match = np.array([np.argmin(((MU_TRUE - c) ** 2).sum(axis=1)) for c in cen_km])  # name k-means clusters like the
lab_km, cen_km = match[lab_km], cen_km[np.argsort(match)]                          # true components (colours match)
hard = pd.DataFrame(np.eye(3, dtype=int)[lab_km[:6]], columns=["cluster 1", "cluster 2", "cluster 3"],
                    index=[f"individual {i + 1}" for i in range(6)])
print("k-means hard assignment:\n", hard)

In [ ]:
# ── 1.3 The same samples with soft assignments (computed from the true model — Section 3 shows how) ──
dens = np.column_stack([PI_TRUE[k] * stats.multivariate_normal(MU_TRUE[k], SIGMA_TRUE[k]).pdf(X) for k in range(3)])
soft = dens / dens.sum(axis=1, keepdims=True)                         # each row sums to 1
table = pd.DataFrame(soft[:6].round(3), columns=["cluster 1", "cluster 2", "cluster 3"],
                     index=[f"individual {i + 1}" for i in range(6)])
table["sum"] = table.sum(axis=1).round(3)
print("probabilistic soft assignment:\n", table)
uncertain = soft.max(axis=1) < 0.9
print(f"\n{uncertain.sum()} of {len(X)} samples have no cluster probability above 0.9")
print(f"k-means' hard label differs from the most probable cluster under the true model for {np.sum(lab_km != soft.argmax(1))} samples")

rgb = np.array([to_rgb(c) for c in CLASS_COLORS[:3]])
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharex=True, sharey=True)
axes[0].scatter(*X.T, c=rgb[lab_km], s=18, edgecolor="k", lw=0.2)
axes[0].scatter(*cen_km.T, marker="X", s=120, c="k")
axes[0].set_title("k-means: hard 0/1 assignment (SIMULATED)")
axes[1].scatter(*X.T, c=soft @ rgb, s=18, edgecolor="k", lw=0.2)             # colours mixed by probability
axes[1].scatter(*X[uncertain].T, s=90, facecolor="none", edgecolor="k", lw=1)
axes[1].set_title("soft assignment: colours mixed by probability\n(circled: no probability above 0.9)")
for ax in axes:
    ax.set_xlabel("expression in condition 1 ($x_1$)")
axes[0].set_ylabel("expression in condition 2 ($x_2$)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The hard table has a single 1 per row; the soft table spreads each row over the three
clusters and every row sums to 1 — individual 4, for example, is 80 % cluster 2 and 20 % cluster 3. Most
samples are nearly certain, but 15 of the 150 (circled) have no cluster probability above 0.9: most of them sit in the
gap between the tall red group and the diagonal green group, a few at the lower edge of the blue group, and their colours
are visibly mixed. k-means painted these points just as confidently as the ones sitting on a centre — and for a few of
them its hard label even differs from the most probable cluster under the true model, because k-means only knows round
distances to centres. That information about **uncertainty** is what soft clustering adds: a point deep inside cluster 2
gets a vector like (0, 1, 0), a point on the border between clusters 1 and 2 one like (0.3, 0.7, 0).

Notice also that the three groups have **different shapes** — compact, tall, diagonal. A Gaussian mixture models each
cluster with its own multivariate Gaussian $N(\boldsymbol\mu_i, \Sigma_i)$ — in two dimensions a $2\times1$ mean vector and a
$2\times2$ covariance matrix per cluster. k-means, which only knows centres and Euclidean distance, cannot represent
shapes at all. In short: *mixture models are a generalisation of k-means clustering* with "soft" probabilistic instead of
"hard" assignments.

✏️ **Your turn.** Change the threshold `0.9` in cell 1.3 to `0.99`. How many samples count as "uncertain" now? Is there
a natural threshold, or is uncertainty a matter of degree?

### 📘 Lecture notes — clustering, hard and soft

**Clustering** takes $N$ samples, each a vector of $J$ variables $\mathbf x^n=[x^n_1,\dots,x^n_J]$, finds $K$ groups of
similar samples and reorders the rows of the $N\times J$ data table so that the groups become visible blocks (gene
time courses summarised by centroid curves; genotypes grouped into populations).

**Hard assignment (k-means).** An $N\times K$ table with exactly one 1 per row (e.g. individual 1: 1, 0, 0).
It cannot express that a point near a centre is *certain* while a point between two centres is *uncertain*.

**Soft (probabilistic) assignment.** The table holds probabilities $P(c^n = i\mid\mathbf x^n)$: every entry is between 0
and 1 and **every row sums to 1**, e.g. individual 1: (0.1, 0.4, 0.5); individual 2: (0.8, 0.1, 0.1). Each cluster is
described by a centre (its mean) and, in a **Gaussian mixture model (GMM)**, also by its own covariance — in two
dimensions $\mathcal N(\boldsymbol\mu_i,\Sigma_i)$ with $\boldsymbol\mu_i$ a $2\times1$ vector and $\Sigma_i$ a $2\times2$
matrix — so clusters can be tilted, elongated or of different sizes.

**Key message.** Mixture models generalise k-means (soft assignments, cluster shapes). Plan: the probability
**model**, **inference** (labels from data) and **learning** (MLE with observed labels; EM, with inference as a
subroutine, without).

> 🎓 **Exam tip.** Know the defining property of the soft table — non-negative entries, each row sums to 1 — and be able to explain
    in one sentence why a hard assignment loses information.

---
## 2 · The Gaussian mixture model

> 📍 **Where we are.** The lecture on mixture models has three parts: the **probability model**, **inference** (infer
> cluster labels from data) and **learning** (MLE with fully observed data; EM with partially observed data, using
> inference as a subroutine). This section is the model.

🧮 **Math refresher — the Gaussian, in one and in $J$ dimensions.** In [Module 02](02_probability_for_biologists.ipynb)
you met the normal (Gaussian) density of one variable with mean $\mu$ and standard deviation $\sigma$:

$$\mathcal N(x\mid\mu,\sigma^2)=\frac{1}{\sqrt{2\pi\sigma^2}}\exp\!\Big(-\frac{(x-\mu)^2}{2\sigma^2}\Big).$$

*In words:* a bell curve centred at $\mu$ whose width is set by $\sigma$. For $J$ variables at once we use the
**multivariate Gaussian** with a **mean vector** $\boldsymbol\mu$ ($J\times1$) and a **covariance matrix** $\Sigma$ ($J\times J$):

$$\mathcal N(\mathbf x\mid\boldsymbol\mu,\Sigma)=\frac{1}{(2\pi)^{J/2}\,|\Sigma|^{1/2}}
\exp\!\Big(-\tfrac12(\mathbf x-\boldsymbol\mu)^\top\Sigma^{-1}(\mathbf x-\boldsymbol\mu)\Big).$$

*In words:* the diagonal entries of $\Sigma$ are the variances of the individual variables, the off-diagonal entries their
covariances (how much they move together — this tilts the bell). The **quadratic form**
$(\mathbf x-\boldsymbol\mu)^\top\Sigma^{-1}(\mathbf x-\boldsymbol\mu)$ is a squared distance from the centre that takes the
shape of the cloud into account (the *Mahalanobis* distance); $|\Sigma|$, the **determinant**, measures the volume of the
cloud and keeps the total probability equal to 1; $\Sigma^{-1}$ is the matrix **inverse** ([Module 03](03_linear_algebra_for_data.ipynb)).
With $J = 1$ the formula reduces to the one-variable bell curve.

**The generative story.** Each data point has a **latent** (hidden) cluster label $c\in\{1,\dots,K\}$ and an
observed value $\mathbf x\in\mathbb R^J$, produced in two steps:

- **S1 — cluster label:** $p(c = i) = \pi_i$ (a *categorical* distribution: a $K$-sided die with face probabilities
  $\pi_1,\dots,\pi_K$, $\sum_i\pi_i = 1$);
- **S2 — data given label:** $\mathbf x \mid c = i \sim \mathcal N(\boldsymbol\mu_i, \Sigma_i)$.

For three clusters: $p(c=1)=\pi_1,\ p(c=2)=\pi_2,\ p(c=3)=\pi_3$ in step S1, then $\mathbf x\mid c=1\sim\mathcal N(\boldsymbol\mu_1,\Sigma_1)$,
$\mathbf x\mid c=2\sim\mathcal N(\boldsymbol\mu_2,\Sigma_2)$ or $\mathbf x\mid c=3\sim\mathcal N(\boldsymbol\mu_3,\Sigma_3)$ in step S2.
The **overall distribution** of $\mathbf x$ sums over the label we never see:

$$p(\mathbf x)=\sum_{i=1}^{K}p(\mathbf x, c=i)=\sum_{i=1}^{K}\underbrace{p(c=i)}_{\pi_i\ \text{(mixing proportion)}}\;
\underbrace{p(\mathbf x\mid c=i)}_{\mathcal N(\mathbf x\mid\boldsymbol\mu_i,\Sigma_i)\ \text{(mixture component)}} .$$

*In words:* the density of a mixture is a weighted average of $K$ bell-shaped components; the weights $\pi_i$ are the
relative cluster sizes (mixing proportion = cluster size). The parameters of a Gaussian mixture are the **mixing
proportions** $p(c = i) = \pi_i$ and the **mixture-component parameters** — mean $\boldsymbol\mu_i$ and covariance
$\Sigma_i$ — for $i = 1, \dots, K$.

In [ ]:
# ── 2.1 Gaussian densities from scratch, checked against SciPy ──
def normal_pdf(x, mu, sd):
    """One-dimensional Gaussian density N(x | mu, sd²)."""
    return np.exp(-0.5 * ((x - mu) / sd) ** 2) / np.sqrt(2 * np.pi * sd ** 2)


def mvn_logpdf(X, mu, Sigma):
    """log N(x | mu, Sigma) for every row x of X (N × J), using a stable log-determinant and a linear solve."""
    X = np.atleast_2d(X)
    J = X.shape[1]
    diff = X - mu
    _, logdet = np.linalg.slogdet(Sigma)                                # log |Σ|
    maha = np.sum(diff * np.linalg.solve(Sigma, diff.T).T, axis=1)      # (x − μ)ᵀ Σ⁻¹ (x − μ)
    return -0.5 * (J * np.log(2 * np.pi) + logdet + maha)


print("1-D:", np.isclose(normal_pdf(1.3, 0.5, 0.8), stats.norm.pdf(1.3, 0.5, 0.8)))
print("2-D:", np.allclose(mvn_logpdf(X[:5], MU_TRUE[2], SIGMA_TRUE[2]),
                          stats.multivariate_normal(MU_TRUE[2], SIGMA_TRUE[2]).logpdf(X[:5])))

Mixtures are useful even in one dimension. A standard example in Bishop's *Pattern Recognition and Machine Learning*
(2006) is the eruption durations of the Old Faithful geyser, whose histogram has two bumps (short and long
eruptions): one Gaussian (unimodal) puts its peak between the bumps, a mixture of two Gaussians (multimodal) follows
both — **a density model $p(x)$ may need to be multi-modal**.
Our biological version (SIMULATED): the log₁₀ expression of a marker gene in 400 cells, where 65 % of the cells have the
gene "off" (mean 2.0) and 35 % have it "on" (mean 4.0). We generate the cells exactly by steps S1 and S2 — this is
called **ancestral sampling**: sample the parent ($c$) first, then the child ($x$).

In [ ]:
# ── 2.2 A one-dimensional mixture: a marker gene that is off in most cells (SIMULATED) ──
rng = np.random.default_rng(SEED + 22)
PI_1D, MU_1D, SD_1D = np.array([0.65, 0.35]), np.array([2.0, 4.0]), np.array([0.5, 0.7])
c_1d = rng.choice(2, size=400, p=PI_1D)                     # S1: is the gene off (0) or on (1) in this cell?
x_1d = rng.normal(MU_1D[c_1d], SD_1D[c_1d])                  # S2: expression given the state
print(f"cells 'on': {np.mean(c_1d == 1):.2f} (true proportion 0.35)")

grid = np.linspace(0, 6.5, 400)
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.hist(x_1d, bins=40, density=True, color="0.85", edgecolor="0.6", label="400 cells (SIMULATED)")
for k, name in enumerate(["off", "on"]):
    ax.plot(grid, PI_1D[k] * normal_pdf(grid, MU_1D[k], SD_1D[k]), "--", color=CLASS_COLORS[k],
            label=f"component '{name}': π = {PI_1D[k]}")
ax.plot(grid, sum(PI_1D[k] * normal_pdf(grid, MU_1D[k], SD_1D[k]) for k in range(2)), color="k", lw=2,
        label="mixture p(x) — bimodal")
ax.plot(grid, normal_pdf(grid, x_1d.mean(), x_1d.std()), color="tab:orange", lw=1.5,
        label="one Gaussian fitted to all cells")
ax.set_xlabel("marker-gene expression (log10)"); ax.set_ylabel("density"); ax.legend(fontsize=7)
ax.set_title("a two-component mixture is multimodal; one Gaussian is not")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The histogram has two bumps. The mixture density (black) — the sum of the two weighted
components (dashed) — follows both, while the single Gaussian (orange) puts its peak in the valley between them, where
almost no cells are. Each dashed component integrates to its $\pi_i$, so the black curve integrates to 1.

Now the two-dimensional version used throughout this notebook — the generating model of Section 1.

In [ ]:
# ── 2.3 The 2-D mixture behind Section 1: components and overall density ──
def draw_ellipse(ax, mu, Sigma, color, n_sd=2.0, **kw):
    """Draw the n_sd-standard-deviation contour ellipse of a 2-D Gaussian."""
    vals, vecs = np.linalg.eigh(Sigma)
    angle = np.degrees(np.arctan2(vecs[1, -1], vecs[0, -1]))
    ax.add_patch(Ellipse(mu, 2 * n_sd * np.sqrt(vals[-1]), 2 * n_sd * np.sqrt(vals[0]), angle=angle,
                         fill=False, color=color, lw=2, **kw))


g1, g2 = np.meshgrid(np.linspace(-0.5, 5.5, 200), np.linspace(-0.5, 5.5, 200))
XY_grid = np.c_[g1.ravel(), g2.ravel()]
mix_density = sum(PI_TRUE[k] * np.exp(mvn_logpdf(XY_grid, MU_TRUE[k], SIGMA_TRUE[k])) for k in range(3)).reshape(g1.shape)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
for k in range(3):
    axes[0].scatter(*X[c_true == k].T, s=12, color=CLASS_COLORS[k], label=f"c = {k + 1} (π = {PI_TRUE[k]})")
    for n_sd in (1, 2):
        draw_ellipse(axes[0], MU_TRUE[k], SIGMA_TRUE[k], CLASS_COLORS[k], n_sd=n_sd)
axes[0].legend(fontsize=7, loc="upper left"); axes[0].set_title("S1 + S2: labels and component ellipses (1 and 2 sd)")
cs = axes[1].contourf(g1, g2, mix_density, levels=15, cmap="viridis")
axes[1].scatter(*X.T, s=4, color="w")
plt.colorbar(cs, ax=axes[1], label="p(x)")
axes[1].set_title("overall distribution p(x) = Σᵢ πᵢ N(x | μᵢ, Σᵢ)")
for ax in axes:
    ax.set_xlabel("$x_1$"); ax.set_aspect("equal")
axes[0].set_ylabel("$x_2$")
fig.suptitle("the three-cluster Gaussian mixture of this notebook (SIMULATED)", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Left: each sample coloured by the label $c$ it was born with, and each component drawn as
contour ellipses — the shape encoded by $\Sigma_i$. Right: the overall density $p(\mathbf x)$ is a landscape with three
hills of different heights and shapes — seen from above, a mountain range with three peaks. In real data we only ever see
the white dots on the right — not the colours on the left.

### A mixture model is a tiny Bayesian network

A mixture model is a special case of a graph with two nodes, $c \rightarrow x$ ($c$: cluster label, $x$: observed data): a **Bayesian network** (the topic of
[Modules 15–17](15_bayesian_networks_representation.ipynb)) whose joint distribution factorises as

$$P(x, c) = P(c)\,P(x\mid c), \qquad P(x)=\sum_{i=1}^{K}P(x\mid c=i)\,P(c=i)=\sum_{i=1}^{K}P(x, c=i).$$

*In words:* first the parent $c$, then the child $x$ given its parent; to get the distribution of what we observe, sum
the joint over the hidden parent. Looking ahead: a **hidden Markov model** is "many mixture models threaded over time" —
each hidden state $c_t$ depends on the previous one $c_{t-1}$ ([Module 19](19_hmm_inference.ipynb)).

In [ ]:
# ── 2.4 The two graphs, and how many parameters a GMM has ──
fig, axes = plt.subplots(1, 2, figsize=(9, 2.6), gridspec_kw=dict(width_ratios=[1, 3]))
draw_graph([("c", "x")], {"c": (0.5, 0.85), "x": (0.5, 0.15)}, ax=axes[0], highlight={"x"},
           title="mixture model c → x")
pos = {f"c{t}": (0.1 + 0.27 * t, 0.8) for t in range(4)} | {f"x{t}": (0.1 + 0.27 * t, 0.2) for t in range(4)}
edges = [(f"c{t}", f"x{t}") for t in range(4)] + [(f"c{t}", f"c{t + 1}") for t in range(3)]
draw_graph(edges, pos, ax=axes[1], highlight={f"x{t}" for t in range(4)},
           title="hidden Markov model = mixtures threaded over time")
plt.tight_layout(); plt.show()


def n_params_gmm(K: int, J: int, cov: str = "full") -> int:
    """Free parameters of a K-component GMM in J dimensions: weights + means + covariances."""
    n_cov = {"full": J * (J + 1) // 2, "diag": J, "spherical": 1}[cov]
    return (K - 1) + K * J + K * n_cov


for J in [2, 50, 500, 3571]:
    print(f"K = 3, J = {J:5d} variables: full covariance {n_params_gmm(3, J):>12,d} parameters | "
          f"diagonal {n_params_gmm(3, J, 'diag'):>7,d} | spherical {n_params_gmm(3, J, 'spherical'):>6,d}")

🔍 **What just happened?** The orange node $x$ is observed, the blue node $c$ is hidden — the same convention we will use
for Bayesian networks and HMMs. The parameter count is

$$\underbrace{(K-1)}_{\pi\text{'s (they sum to 1)}}+\underbrace{KJ}_{\text{means}}+\underbrace{K\,\tfrac{J(J+1)}{2}}_{\text{symmetric covariance matrices}} .$$

*In words:* with two variables a 3-component GMM has 17 parameters; with the 3,571 genes of a leukaemia expression
data set a full-covariance GMM would need about 19 million — hopeless with 72 patients. Restricting each covariance to be
**diagonal** (genes independent within a cluster) or **spherical** ($\sigma^2 I$) is what makes mixtures usable in high
dimensions (Section 10) — and the spherical case is the bridge to k-means (Section 6).

✏️ **Your turn.** Change `PI_1D` in cell 2.2 to `[0.95, 0.05]` (a rare "on" population) and re-run. Is the second mode
still visible in the histogram? Rare cell types are hard to see in a density — but a mixture model can still give them
their own component.

### 📘 Lecture notes — the Gaussian mixture model

**Model summary.** Data $\mathbf x^1,\dots,\mathbf x^N\in\mathbb R^J$ are generated independently (i.i.d.) as

- (latent variable) cluster label $c^n\sim\mathrm{Categorical}(\pi_1,\dots,\pi_K)$, i.e. $p(c^n=i)=\pi_i$, $\sum_i\pi_i=1$;
- (observed) $\mathbf x^n\mid c^n=i\ \sim\ \mathcal N(\boldsymbol\mu_i,\Sigma_i)$;
- parameters: $\pi_i$ (mixing proportion = cluster size), $\boldsymbol\mu_i$ ($J\times1$) and $\Sigma_i$ ($J\times J$) for
  each component $i$;
- distribution of one data point:
  $$p(\mathbf x^n)=\sum_{i=1}^{K}p(c^n=i)\,p(\mathbf x^n\mid c^n=i)=\sum_{i=1}^{K}\pi_i\,\mathcal N(\mathbf x^n\mid\boldsymbol\mu_i,\Sigma_i).$$
  *In words:* a weighted sum of $K$ Gaussian "mixture components", with the cluster sizes as weights.

**As a Bayesian network** $c\to x$: $P(x,c)=P(c)\,P(x\mid c)$ and $P(x)=\sum_i P(x, c=i)=\sum_i P(x\mid c=i)P(c=i)$.
An HMM chains such mixtures over time: $c_1\to c_2\to\dots\to c_T$, each $c_t\to x_t$.

**Why mixtures?** A single Gaussian is unimodal; a mixture can be **multimodal** and give each cluster its own shape.

**Parameter count** of a $K$-component GMM in $J$ dimensions with full covariances:
$(K-1)+KJ+K\,J(J+1)/2$ — for $K=3$, $J=2$: $2+6+9=17$.

> 🎓 **Exam tip.** Be able to write p(x) = Σᵢ πᵢ N(x | μᵢ, Σᵢ), say what each symbol is (and its dimension: μᵢ is J × 1, Σᵢ is J × J),
    and count the free parameters of a GMM (remember the K − 1 for the π's and J(J+1)/2 per symmetric covariance).

---
## 3 · Inference: who belongs where?

> 📍 **Where we are.** Suppose the model — all $\pi_i, \boldsymbol\mu_i, \Sigma_i$ — is known, say a mixture model
> already estimated from data. Given a sample $\mathbf x$, what is the probability that it came from cluster $i$?
> This is **inference** in the network $c\to x$: reasoning from the observed child back to the hidden parent.

Start from the joint distribution $p(\mathbf x, c=i)=p(\mathbf x\mid c=i)\,p(c=i)$ and apply **Bayes' rule**
([Module 02](02_probability_for_biologists.ipynb)):

$$p(c=i\mid\mathbf x)=\frac{p(\mathbf x\mid c=i)\,p(c=i)}{p(\mathbf x)}
=\frac{\pi_i\,\mathcal N(\mathbf x\mid\boldsymbol\mu_i,\Sigma_i)}{\sum_{j=1}^{K}\pi_j\,\mathcal N(\mathbf x\mid\boldsymbol\mu_j,\Sigma_j)} .$$

*In words:* for each cluster multiply "how big is the cluster" ($\pi_i$, the **prior**) by "how well does the cluster
explain this point" (the **likelihood** $\mathcal N(\mathbf x\mid\cdot)$), then divide by the total so that the $K$
numbers sum to 1. The result is the **posterior** probability of each cluster — often called the **responsibility** of
cluster $i$ for sample $n$, written $\gamma_{ni} = p(c^n = i\mid\mathbf x^n)$. The denominator is the same for every $i$,
so it does not matter if you only want the **most probable** cluster — for each $i$ compute
$p(\mathbf x^n\mid c^n=i)\,p(c^n=i)$ and pick the largest — but it is needed if you want actual probabilities, for
example to fill in the soft-assignment table of Section 1, where
$P(c=1\mid\mathbf x)+P(c=2\mid\mathbf x)+P(c=3\mid\mathbf x)=1$ for every individual.

In [ ]:
# ── 3.1 Bayes' rule by hand for one sample ──
n = int(np.argmax(soft.max(axis=1) < 0.9))                # the first "uncertain" sample of Section 1
x_n = X[n]
print(f"sample {n + 1}: x = {np.round(x_n, 2)}")
prior = PI_TRUE
lik = np.array([np.exp(mvn_logpdf(x_n, MU_TRUE[i], SIGMA_TRUE[i]))[0] for i in range(3)])
joint = prior * lik
for i in range(3):
    print(f"  cluster {i + 1}: prior π = {prior[i]:.2f} × likelihood N(x|μ,Σ) = {lik[i]:.4f}  →  joint = {joint[i]:.5f}")
print(f"  p(x) = sum of joints = {joint.sum():.5f}")
print("  posterior p(c = i | x) =", np.round(joint / joint.sum(), 3), "| sums to", round((joint / joint.sum()).sum(), 6))

🔍 **What just happened?** This sample (individual 4 of the tables) lies between the tall cluster 2 and the diagonal
cluster 3. Cluster 2 explains it better (likelihood 0.34 vs 0.06), but cluster 3 is larger ($\pi_3 = 0.4$ vs $0.3$), so
the posterior is split 0.805 / 0.195; cluster 1, far away, gets essentially nothing. Dividing by $p(\mathbf x)$ turned
three joint numbers into probabilities that sum to 1.

⚠️ **Numerical underflow.** With many variables, likelihoods become astronomically small: the density of a point in
$J = 1{,}000$ dimensions is of order $e^{-1400}$, far below the smallest positive number a computer can store
($\approx 10^{-308}$), so it is rounded to 0 and Bayes' rule becomes $0/0$. The cure is to work with **logarithms** and
the **log-sum-exp** trick ([Module 04](04_calculus_and_optimization.ipynb), Section 3; toolkit `logsumexp`):

$$\log\sum_j e^{a_j}=m+\log\sum_j e^{a_j-m},\qquad m=\max_j a_j,
\qquad \log\gamma_{ni} = a_i - \log\sum_j e^{a_j}\ \ \text{with}\ a_j=\log\pi_j+\log\mathcal N(\mathbf x^n\mid\boldsymbol\mu_j,\Sigma_j).$$

*In words:* subtract the largest log-number before exponentiating, so that the biggest term becomes $e^0 = 1$ and
nothing underflows; add it back afterwards.

In [ ]:
# ── 3.2 Responsibilities for all samples, in log space ──
def responsibilities(X, pis, mus, Sigmas):
    """E-step / inference: returns (gamma N × K with gamma[n, i] = p(c = i | x_n), total log-likelihood)."""
    log_joint = np.column_stack([np.log(pis[i]) + mvn_logpdf(X, mus[i], Sigmas[i]) for i in range(len(pis))])
    log_px = logsumexp(log_joint, axis=1)                    # log p(x_n) = log Σ_i π_i N(x_n | μ_i, Σ_i)
    return np.exp(log_joint - log_px[:, None]), float(log_px.sum())


gamma_true, ll_true = responsibilities(X, PI_TRUE, MU_TRUE, SIGMA_TRUE)
tab = pd.DataFrame(gamma_true[:10].round(3), columns=["cluster 1", "cluster 2", "cluster 3"],
                   index=[f"individual {i + 1}" for i in range(10)])
tab["sum"] = tab.sum(axis=1).round(6)
print("the soft-assignment table, populated by inference:\n", tab)
print("\nsame as the direct computation of Section 1:", np.allclose(gamma_true, soft))
map_label = gamma_true.argmax(axis=1)                         # hard decision: the most probable cluster
print(f"most-probable cluster = true generating cluster for {np.mean(map_label == c_true):.1%} of samples")

# the underflow problem in 1,000 dimensions
rng = np.random.default_rng(SEED + 32)
x_big = rng.normal(size=(1, 1000))
logs = np.array([mvn_logpdf(x_big, np.zeros(1000), np.eye(1000))[0], mvn_logpdf(x_big, 0.1 * np.ones(1000), np.eye(1000))[0]])
with np.errstate(invalid="ignore"):
    naive = np.exp(logs) / np.exp(logs).sum()
print(f"\nJ = 1000: log-densities {np.round(logs, 1)} → exp gives {np.exp(logs)} → naive posterior {naive}")
print("log-sum-exp posterior:", np.round(np.exp(logs - logsumexp(logs)), 4))

🔍 **What just happened?** Every row of the inferred table sums to 1, as it must
($P(c=1\mid x)+P(c=2\mid x)+P(c=3\mid x)=1$), and the log-space computation reproduces Section 1. Picking the most
probable cluster recovers the true generating label for most samples; the few "errors" are points that genuinely sit
inside another cluster's territory — no method could get them right. In 1,000 dimensions the naive computation returns
`nan` (0/0), while log-sum-exp gives sensible probabilities.

Let us map the posterior over the whole plane.

In [ ]:
# ── 3.3 Where is the model certain? ──
gamma_grid, _ = responsibilities(XY_grid, PI_TRUE, MU_TRUE, SIGMA_TRUE)
fig, axes = plt.subplots(1, 2, figsize=(10, 3.9), sharex=True, sharey=True)
axes[0].contourf(g1, g2, gamma_grid.argmax(1).reshape(g1.shape), levels=[-0.5, 0.5, 1.5, 2.5],
                 colors=CLASS_COLORS[:3], alpha=0.25)
axes[0].scatter(*X.T, c=rgb[c_true], s=10)
axes[0].set_title("most probable cluster (hard decision)")
cs = axes[1].contourf(g1, g2, gamma_grid.max(1).reshape(g1.shape), levels=np.linspace(1 / 3, 1, 15), cmap="magma")
axes[1].scatter(*X.T, s=4, color="w")
plt.colorbar(cs, ax=axes[1], label="max_i p(c = i | x)")
axes[1].set_title("certainty of the assignment")
for ax in axes:
    ax.set_xlabel("$x_1$"); ax.set_aspect("equal")
axes[0].set_ylabel("$x_2$")
fig.suptitle("posterior cluster probabilities under the true model (SIMULATED)", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Left: the plane is divided into three regions by the most probable cluster. Because the
clusters have different covariances, the boundaries are **curved** (quadratic), unlike the straight Voronoi boundaries of
k-means. Right: certainty is near 1 (bright) inside the clusters and drops in bands along the boundaries — the
"uncertain" zones of Section 1, now quantified. Far from all data the model can still be very certain (whichever component
has the widest tails wins): a posterior probability is a statement *within the model*, not a guarantee.

✏️ **Your turn.** Re-run 3.3 with `PI_TRUE` replaced by `np.array([0.3, 0.05, 0.65])` in the call to `responsibilities`.
How do the boundaries around cluster 2 move? A small prior makes the model demand more evidence before it calls a point
"cluster 2" — the same prior effect you saw in naive Bayes ([Module 08](08_knn_naive_bayes.ipynb)).

### 📘 Lecture notes — inference in a mixture model

**Question.** The model ($\pi_i,\boldsymbol\mu_i,\Sigma_i$ for each component $i$) is known. Given a sample
$\mathbf x^n$, which cluster did it come from? In the graphical model $c\to x$: observe $x$, infer $c$.

**Bayes' rule.**
$$p(c^n=i\mid\mathbf x^n)=\frac{p(\mathbf x^n\mid c^n=i)\,p(c^n=i)}{\sum_{j=1}^{K}p(\mathbf x^n\mid c^n=j)\,p(c^n=j)}
=\frac{\pi_i\,\mathcal N(\mathbf x^n\mid\boldsymbol\mu_i,\Sigma_i)}{\sum_j\pi_j\,\mathcal N(\mathbf x^n\mid\boldsymbol\mu_j,\Sigma_j)} .$$
*In words:* posterior ∝ prior (cluster size) × likelihood (fit of the point to the cluster), normalised over clusters.

**Two uses.** (1) *Hard decision* — the denominator is the same for every $i$, so for each $i$ compute
$p(\mathbf x^n\mid c^n=i)\,p(c^n=i)$ and select the $i$ with the largest value. (2) *Soft assignment* — divide by the
denominator to get the probabilities that populate the $N\times K$ table, each row summing to 1; these
**responsibilities** $\gamma_{ni}$ are what the E-step of EM needs.

**Worked example.** Three clusters with $\pi=(0.5, 0.3, 0.2)$ and likelihoods $\mathcal N(\mathbf x\mid\cdot)=(0.02, 0.10, 0.25)$:
joint terms $0.010, 0.030, 0.050$; $p(\mathbf x)=0.090$; posterior $(0.11, 0.33, 0.56)$ — the smallest cluster wins
because it explains the point best; the row sums to 1.

**In practice** compute in logs with the log-sum-exp trick: high-dimensional densities underflow to 0.

> 🎓 **Exam tip.** Know when the denominator matters: not for the arg max (hard decision), but yes for the probabilities themselves
    (the soft table and the E-step of EM). Be able to compute a 2- or 3-cluster posterior by hand from given πᵢ and
    likelihood values.

---
## 4 · Learning when the labels are observed

> 📍 **Where we are.** Inference assumed the parameters were known. Where do they come from? Learning has two cases:
> **fully observed data** $(c, x)$ — use maximum likelihood; **partially observed data** — only $x$, no observations of
> $c$ — use the EM algorithm (Section 5); and on top of both, model selection — how many clusters (Section 8)? First the
> easy case: suppose that for every sample we also observe its cluster label $c^n$ — say, a pathologist has typed
> every patient. Then learning is ordinary maximum likelihood ([Module 05](05_likelihood_mle_map.ipynb)).

With independent samples the log-likelihood of the pairs $(\mathbf x^n, c^n)$ is

$$\log p\big((\mathbf x^1,c^1),\dots,(\mathbf x^N,c^N)\big)=\sum_{n=1}^{N}\log p(\mathbf x^n\mid c^n)p(c^n)
=\underbrace{\sum_{n=1}^{N}\log p(\mathbf x^n\mid c^n)}_{\text{depends only on }\boldsymbol\mu_i,\Sigma_i}
+\underbrace{\sum_{n=1}^{N}\log p(c^n)}_{\text{depends only on }\pi_i}.$$

*In words:* the log turns the product into a sum, and the sum splits into a part about the components and a part about
the proportions — they can be optimised separately, which makes the problem easy. The maxima are:

$$\pi_i=\frac{N_i}{N},\qquad
\boldsymbol\mu_i=\frac{\sum_{n=1}^{N}\mathbf x^n\,I(c^n=i)}{N_i},\qquad
\Sigma_i=\frac{\sum_{n=1}^{N}(\mathbf x^n-\boldsymbol\mu_i)(\mathbf x^n-\boldsymbol\mu_i)^\top I(c^n=i)}{N_i},
\qquad N_i=\sum_{n=1}^{N}I(c^n=i).$$

*In words:* the **indicator** $I(c^n = i)$ is 1 if sample $n$ is in cluster $i$ and 0 otherwise, so $N_i$ counts the
cluster's members. The mixing proportion is the fraction of samples in the cluster; the mean is the average of its
members; the covariance is the average outer product of the members' deviations from that mean.

🧮 **Why $\pi_i = N_i/N$?** We must maximise $\sum_n\log p(c^n)=\sum_i N_i\log\pi_i$ subject to $\sum_i\pi_i = 1$. This is
the dice problem of Module 05: the MLE of a face probability is its observed frequency. (Formally, a Lagrange multiplier
$\lambda$ for the constraint gives $N_i/\pi_i=\lambda$, so $\pi_i\propto N_i$, and normalising gives $\lambda = N$.) The mean
and covariance formulas are the single-Gaussian MLEs of Module 05, applied to each cluster's members. Note the
denominator $N_i$, not $N_i - 1$: the MLE of a variance is slightly biased, which hardly matters for large clusters.

In [ ]:
# ── 4.1 MLE with observed labels ──
def fit_labelled(X, c, K):
    """MLE of a GMM when every sample's cluster label c is known: counting and averaging."""
    pis = np.array([np.mean(c == i) for i in range(K)])
    mus = np.array([X[c == i].mean(axis=0) for i in range(K)])
    Sigmas = np.array([(X[c == i] - mus[i]).T @ (X[c == i] - mus[i]) / np.sum(c == i) for i in range(K)])
    return pis, mus, Sigmas


pi_hat, mu_hat, Sigma_hat = fit_labelled(X, c_true, 3)
print("π  estimated:", pi_hat.round(3), "  true:", PI_TRUE)
print("μ  estimated:", mu_hat.round(2).tolist(), "\n   true:     ", MU_TRUE.tolist())
print("Σ₃ estimated:", Sigma_hat[2].round(3).tolist(), "  true:", SIGMA_TRUE[2].tolist())
print("Σ matches np.cov(..., bias=True):", all(np.allclose(Sigma_hat[i], np.cov(X[c_true == i].T, bias=True)) for i in range(3)))

🔍 **What just happened?** With the labels in hand, counting and averaging recover the generating parameters closely —
they differ only by sampling noise (covariances, estimated from 44–61 members per cluster, are the noisiest) — and our
covariance is NumPy's covariance with the $1/N_i$ convention. This is learning with **fully observed data**: one pass
through the data, no iterations.

✏️ **Your turn.** Call `fit_labelled(X[:30], c_true[:30], 3)`. How far are the estimates from the truth with only 30
labelled samples? Which parameters suffer most?

### 📘 Lecture notes — learning with fully observed data

**Data** $(\mathbf x^1,c^1),\dots,(\mathbf x^N,c^N)$ — every sample with its cluster label. **Goal:** learn
$\pi_i,\boldsymbol\mu_i,\Sigma_i$ for $i=1,\dots,K$ by maximum likelihood.

**Key step.**
$$\log\prod_{n}p(\mathbf x^n,c^n)=\sum_n\log p(\mathbf x^n\mid c^n)+\sum_n\log p(c^n).$$
*In words:* the first sum involves only the Gaussian components, the second only the mixing proportions, so the two
parts are maximised separately.

**Procedure (closed form, one pass, no iterations).** With the indicator $I(c^n=i)$ and $N_i=\sum_n I(c^n=i)$:
- mixing proportions $\pi_i=N_i/N$ (fraction of samples in cluster $i$);
- means $\boldsymbol\mu_i=\sum_n\mathbf x^n I(c^n=i)/N_i$ (average of the members);
- covariances $\Sigma_i=\sum_n(\mathbf x^n-\boldsymbol\mu_i)(\mathbf x^n-\boldsymbol\mu_i)^\top I(c^n=i)/N_i$ (average outer
  product of the members' deviations; MLE, so $1/N_i$ rather than $1/(N_i-1)$).

**Tiny example.** One-dimensional data $x=(1.0, 1.4, 3.0, 3.4, 3.2)$ with labels $c=(1,1,2,2,2)$: $\pi=(0.4, 0.6)$,
$\mu_1=1.2$, $\mu_2=3.2$, $\sigma_1^2=0.04$, $\sigma_2^2=(0.04+0.04+0)/3\approx0.027$.

> 🎓 **Exam tip.** The derivation "log of a product = sum of logs, and the sum separates" is the key step to reproduce — it is exactly
    what fails when the labels are hidden (Section 5), because then the log is applied to a sum.

---
## 5 · Labels hidden: the EM algorithm

> 📍 **Where we are.** Now the real situation: we observe $\mathbf x^1,\dots,\mathbf x^N$ but **not** the
> labels — $c$ is a **latent variable** (a *latent-variable model*: $c$ unobserved, only $x$ observed). What we know so
> far: *MLE is easy if labels were observed* (Section 4); *labels can be inferred if the parameters were known*
> (Section 3). Each needs the other — a chicken-and-egg problem.

**What we would like to maximise** is the log-likelihood of what we actually observed, the
**observed-data log-likelihood**:

$$\ell(\theta)=\log\prod_{n=1}^{N}p(\mathbf x^n)=\sum_{n=1}^{N}\log\sum_{i=1}^{K}p(\mathbf x^n\mid c^n=i)\,p(c^n=i),
\qquad \theta=\{\pi_i,\boldsymbol\mu_i,\Sigma_i\}_{i=1}^{K}.$$

*In words:* for each sample, add up the probability of producing it over all clusters it could have come from, then
take logs and sum over samples. The trouble — "difficult to optimise because $c^n$ is missing!" — is the **log of a sum**:
it does not split into separate pieces as in Section 4, and setting derivatives to zero gives equations in which every
parameter depends on all the others.

**The EM idea.** Replace each unknown indicator $I(c^n = i)$ by our current best guess of it, its
**expected value** under the current parameters $\theta^{(t)}$ — which is exactly the responsibility of Section 3:

$$\mathbb E\big[I(c^n=i)\big]=p(c^n=i\mid\mathbf x^n,\theta^{(t)})=\gamma_{ni}.$$

*In words:* since we cannot see whether sample $n$ belongs to cluster $i$ (indicator 0 or 1), we use the probability
that it does — the expected value of a 0/1 variable is the probability that it equals 1.

Then maximise the **expected complete-data log-likelihood** $\;Q(\theta\mid\theta^{(t)})=\mathbb E_{p(c\mid x,\theta^{(t)})}
\big[\log\prod_n p(\mathbf x^n,c^n\mid\theta)\big]=\sum_n\sum_i\gamma_{ni}\log\big[\pi_i\,\mathcal N(\mathbf x^n\mid\boldsymbol\mu_i,\Sigma_i)\big]$,
which *does* split like Section 4. Alternate:

- **E-step** ("assign clusters") — inference with the current parameters $\theta^{(t)}=\{\pi_i,\boldsymbol\mu_i,\Sigma_i\}_{i=1}^K$:
  $$\gamma_{ni}=p(c^n=i\mid\mathbf x^n)=\frac{\pi_i\,\mathcal N(\mathbf x^n\mid\boldsymbol\mu_i,\Sigma_i)}{\sum_{j=1}^{K}\pi_j\,\mathcal N(\mathbf x^n\mid\boldsymbol\mu_j,\Sigma_j)} .$$
- **M-step** ("estimate the mixture parameters") — $\theta^{(t+1)}=\arg\max_\theta Q(\theta\mid\theta^{(t)})$, which gives
  the labelled-data MLEs of Section 4 with $I(c^n=i)$ replaced by $\gamma_{ni}$:
  $$\pi_i=\frac{\sum_n\gamma_{ni}}{N},\qquad
  \boldsymbol\mu_i=\frac{\sum_n\gamma_{ni}\,\mathbf x^n}{\sum_n\gamma_{ni}},\qquad
  \Sigma_i=\frac{\sum_n\gamma_{ni}(\mathbf x^n-\boldsymbol\mu_i)(\mathbf x^n-\boldsymbol\mu_i)^\top}{\sum_n\gamma_{ni}} .$$

*In words:* **E** — let every sample vote for every cluster, with votes proportional to how well the cluster explains it;
**M** — re-estimate each cluster from its (fractional) votes: its size is its total vote, its mean and covariance are
vote-weighted averages. Start by *guessing* the centroid $\boldsymbol\mu_i$ and covariance $\Sigma_i$ of each of the $K$
clusters, then loop E, M, E, M, … until nothing changes.

In [ ]:
# ── 5.1 EM for a one-dimensional Gaussian mixture, from scratch ──
def em_gmm_1d(x, mus, sds, pis, n_iter=500, tol=1e-8, sd_floor=0.0, keep=(0, 1, 3, 10)):
    """EM for a 1-D Gaussian mixture. sd_floor > 0 stops any standard deviation from shrinking below it.

    Returns dict with the final parameters, responsibilities, the log-likelihood trace ℓ(θ⁽ᵗ⁾), the trace of the
    standard deviations and snapshots of the parameters at the iterations listed in `keep`.
    """
    mus, sds, pis = (np.array(a, float) for a in (mus, sds, pis))
    trace, sd_trace, snaps = [], [], {}
    for t in range(n_iter):
        if t in keep:
            snaps[t] = (mus.copy(), sds.copy(), pis.copy())
        with np.errstate(divide="ignore", invalid="ignore", over="ignore"):
            log_joint = np.log(pis) + stats.norm.logpdf(x[:, None], mus, sds)        # N × K
            log_px = logsumexp(log_joint, axis=1)
            trace.append(log_px.sum()); sd_trace.append(sds.copy())
            if not np.isfinite(trace[-1]):
                break                                                             # the likelihood blew up
            gamma = np.exp(log_joint - log_px[:, None])                           # E-step
            Nk = gamma.sum(axis=0)                                                # M-step
            pis = Nk / len(x)
            mus = gamma.T @ x / Nk
            sds = np.maximum(np.sqrt((gamma * (x[:, None] - mus) ** 2).sum(axis=0) / Nk), sd_floor)
        if t > 0 and abs(trace[-1] - trace[-2]) < tol:
            break
    snaps["final"] = (mus, sds, pis)
    return dict(mus=mus, sds=sds, pis=pis, gamma=gamma, trace=np.array(trace), sd_trace=np.array(sd_trace), snaps=snaps)


fit1 = em_gmm_1d(x_1d, mus=[0.5, 1.0], sds=[1.0, 1.0], pis=[0.5, 0.5], keep=(0, 1, 10))   # a deliberately poor start
print(f"converged after {len(fit1['trace'])} iterations")
print("EM estimates: μ =", fit1["mus"].round(3), " σ =", fit1["sds"].round(3), " π =", fit1["pis"].round(3))
print("truth       : μ =", MU_1D, " σ =", SD_1D, " π =", PI_1D)
print("log-likelihood: first values", fit1["trace"][:4].round(2), "… last", fit1["trace"][-1].round(4))
print("never decreased:", bool(np.all(np.diff(fit1["trace"]) >= -1e-9)))
sk1 = GaussianMixture(2, tol=1e-10, max_iter=5000, random_state=SEED).fit(x_1d[:, None])
print(f"scikit-learn log-likelihood: {sk1.score(x_1d[:, None]) * len(x_1d):.4f} (ours {fit1['trace'][-1]:.4f})")

In [ ]:
# ── 5.2 Watch EM converge ──
fig, axes = plt.subplots(1, 5, figsize=(12, 2.7))
for ax, (t, (m, s, p)) in zip(axes[:4], fit1["snaps"].items()):
    ax.hist(x_1d, bins=40, density=True, color="0.88")
    for k in range(2):
        ax.plot(grid, p[k] * normal_pdf(grid, m[k], s[k]), color=CLASS_COLORS[k], lw=1.5)
    ax.plot(grid, sum(p[k] * normal_pdf(grid, m[k], s[k]) for k in range(2)), "k--", lw=1)
    ax.set_title(f"iteration {t}" if t != "final" else f"final (iteration {len(fit1['trace'])})", fontsize=9)
    ax.set_xlabel("expression (log10)"); ax.set_ylim(0, 0.65)
axes[0].set_ylabel("density")
axes[4].plot(np.arange(len(fit1["trace"])), fit1["trace"], "o-", ms=2)
axes[4].set_xlabel("iteration t"); axes[4].set_ylabel("ℓ(θ⁽ᵗ⁾)"); axes[4].set_title("log-likelihood", fontsize=9)
fig.suptitle("EM on the SIMULATED marker gene, from a poor start", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Both components started on the far left, wide and equal. After one E/M round the second
component already moved right, towards the "on" cells; after ten rounds the two components sit on the two populations,
and after 98 iterations the estimates (μ ≈ 1.93 and 3.97, π ≈ 0.65/0.35) are close to the truth and identical to
scikit-learn's answer. The log-likelihood rose
quickly at first and then levelled off — and it **never went down**. Why can we promise that?

### 5.3 Why the likelihood never decreases: the lower bound

🧮 **Math refresher — Jensen's inequality.** The logarithm is **concave** (its graph bends downwards), so the log of an
average is at least the average of the logs: for weights $q_i\ge0$ with $\sum_i q_i=1$ and positive numbers $a_i$,

$$\log\sum_i q_i\,a_i\;\ge\;\sum_i q_i\log a_i ,$$

*In words:* the log of a weighted average is at least the weighted average of the logs, with equality when all $a_i$
are equal. Apply this to one sample's term of $\ell(\theta)$, writing the sum over clusters
as an average with weights $q_i$ (any distribution over clusters):

$$\log\sum_i p(\mathbf x^n, c^n=i\mid\theta)=\log\sum_i q_i\,\frac{p(\mathbf x^n,c^n=i\mid\theta)}{q_i}
\;\ge\;\sum_i q_i\log\frac{p(\mathbf x^n,c^n=i\mid\theta)}{q_i}.$$

*In words:* for **any** soft assignment $q$, the right-hand side — call its sum over samples $B(\theta; q)$ — is a
**lower bound** on the log-likelihood. Two facts make EM work:

1. **E-step = make the bound touch.** Choosing $q_i = \gamma_{ni} = p(c^n=i\mid\mathbf x^n,\theta^{(t)})$ makes all ratios
   $p(\mathbf x^n,c^n=i\mid\theta^{(t)})/q_i$ equal to $p(\mathbf x^n\mid\theta^{(t)})$, so equality holds:
   $B(\theta^{(t)};\gamma)=\ell(\theta^{(t)})$.
2. **M-step = push the bound up.** $B(\theta;\gamma)=Q(\theta\mid\theta^{(t)})-\sum_n\sum_i\gamma_{ni}\log\gamma_{ni}$; the
   second term does not involve $\theta$, so maximising $Q$ (the M-step formulas) maximises $B$.

Chaining them: $\;\ell(\theta^{(t+1)})\;\ge\;B(\theta^{(t+1)};\gamma)\;\ge\;B(\theta^{(t)};\gamma)\;=\;\ell(\theta^{(t)})$.

*In words:* the bound is never above the true curve, touches it at the current parameters, and the M-step climbs to the
top of the bound — so the true curve at the new parameters can only be higher (or equal). Let us draw this at the poor
starting point of 5.1, varying just one parameter ($\mu_2$, the "on" mean) and holding the others fixed.

In [ ]:
# ── 5.3 Draw the EM lower bound along one parameter ──
m_t, s_t, p_t = fit1["snaps"][0]                                     # the poor starting parameters θ⁽⁰⁾
log_joint_t = np.log(p_t) + stats.norm.logpdf(x_1d[:, None], m_t, s_t)
gamma_t = np.exp(log_joint_t - logsumexp(log_joint_t, axis=1)[:, None])   # E-step at θ⁽⁰⁾


def loglik_mu2(mu2):
    """Observed-data log-likelihood ℓ when only μ₂ changes."""
    lj = np.log(p_t) + stats.norm.logpdf(x_1d[:, None], [m_t[0], mu2], s_t)
    return logsumexp(lj, axis=1).sum()


def bound_mu2(mu2):
    """EM lower bound B(θ; γ⁽⁰⁾) = Σₙ Σᵢ γ log(p(x, c = i)/γ) when only μ₂ changes."""
    lj = np.log(p_t) + stats.norm.logpdf(x_1d[:, None], [m_t[0], mu2], s_t)
    return np.sum(gamma_t * (lj - np.log(np.maximum(gamma_t, 1e-300))))


mu2_grid = np.linspace(0.5, 5.5, 200)
L = np.array([loglik_mu2(m) for m in mu2_grid]); B = np.array([bound_mu2(m) for m in mu2_grid])
mu2_new = gamma_t[:, 1] @ x_1d / gamma_t[:, 1].sum()                # M-step for μ₂ (weighted mean)
print(f"μ₂ now {m_t[1]:.3f}: ℓ = {loglik_mu2(m_t[1]):.2f}, bound = {bound_mu2(m_t[1]):.2f} (they touch)")
print(f"M-step μ₂ = {mu2_new:.3f}: bound rises to {bound_mu2(mu2_new):.2f}, ℓ rises to {loglik_mu2(mu2_new):.2f}")
print("bound ≤ ℓ everywhere on the grid:", bool(np.all(B <= L + 1e-9)))

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(mu2_grid, L, color="k", lw=2, label="log-likelihood ℓ(μ₂)")
ax.plot(mu2_grid, B, color="tab:orange", lw=2, label="EM lower bound B(μ₂; γ⁽⁰⁾)")
ax.axvline(m_t[1], color="0.5", ls=":", label=f"current μ₂ = {m_t[1]:.2f}")
ax.axvline(mu2_new, color="tab:green", ls="--", label=f"M-step μ₂ = {mu2_new:.2f}")
ax.set_ylim(loglik_mu2(m_t[1]) - 150, L.max() + 60)
ax.set_xlabel("μ₂ (mean of the 'on' component)"); ax.set_ylabel("log-likelihood")
ax.set_title("the bound touches ℓ at the current value; maximising it moves uphill on ℓ"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The orange bound lies below the black log-likelihood everywhere, touches it at the current
$\mu_2$, and has its maximum at the M-step value (the responsibility-weighted mean). Moving there raises the bound — and
therefore raises $\ell$ by at least as much. The same picture holds in all parameters at once.

### ⚠️ What EM guarantees — stated precisely

EM is often summarised as "guaranteed to converge: the log-likelihood of the observed data increases in each iteration";
it converges to a local optimum, and there can be many local optima, because the latent-variable objective is
non-convex. Precisely:

| | |
|---|---|
| ✅ **Monotonicity** | $\ell(\theta^{(t+1)}) \ge \ell(\theta^{(t)})$ for every iteration: the observed-data log-likelihood **never decreases** (it can stay equal — e.g. at a fixed point — so "increase" means "does not decrease"). |
| ✅ **Convergence of the likelihood values** | if $\ell$ is bounded above, the non-decreasing sequence $\ell(\theta^{(t)})$ converges; under mild conditions the parameters converge to a **stationary point** of $\ell$ — usually a local **maximum** of the likelihood (equivalently, a local *minimum* of the negative log-likelihood, which is how optimisation texts phrase it), occasionally a saddle point (Wu 1983). |
| ❌ **No global guarantee** | which stationary point you reach depends on the starting values (Section 7.1). |
| ❌ **No finite optimum for unconstrained GMMs** | the Gaussian-mixture likelihood is **unbounded**: a component that shrinks onto a single data point has $\ell\to+\infty$. EM can run into such **degenerate solutions**, so for GMMs "guaranteed to converge" needs a constraint such as a variance floor (Section 7.2). |

In [ ]:
# ── 5.4 EM for multivariate Gaussian mixtures (full covariance matrices) ──
def m_step(X, gamma, reg=1e-6):
    """M-step: responsibility-weighted MLEs. reg·I is added to each covariance (a small variance floor)."""
    Nk = gamma.sum(axis=0)
    pis = Nk / len(X)
    mus = gamma.T @ X / Nk[:, None]
    Sigmas = np.array([((gamma[:, i, None] * (X - mus[i])).T @ (X - mus[i])) / Nk[i] + reg * np.eye(X.shape[1])
                       for i in range(gamma.shape[1])])
    return pis, mus, Sigmas


def em_gmm(X, K, init_means=None, n_iter=500, tol=1e-6, reg=1e-6, rng=None, keep=()):
    """EM for a K-component GMM with full covariances. Starts from init_means (or K random samples),
    covariances equal to the data covariance and equal weights. Returns a dict with parameters, gamma, trace, snapshots."""
    rng = rng if rng is not None else np.random.default_rng(0)
    mus = X[rng.choice(len(X), K, replace=False)].copy() if init_means is None else np.array(init_means, float)
    Sigmas = np.array([np.cov(X.T, bias=True)] * K)
    pis = np.full(K, 1.0 / K)
    trace, snaps = [], {}
    for t in range(n_iter):
        if t in keep:
            snaps[t] = (pis.copy(), mus.copy(), Sigmas.copy())
        gamma, ll = responsibilities(X, pis, mus, Sigmas)            # E-step (Section 3)
        trace.append(ll)
        if t > 0 and abs(trace[-1] - trace[-2]) < tol:
            break
        pis, mus, Sigmas = m_step(X, gamma, reg)                      # M-step
    snaps["final"] = (pis, mus, Sigmas)
    return dict(pis=pis, mus=mus, Sigmas=Sigmas, gamma=gamma, trace=np.array(trace), snaps=snaps)


fit2 = em_gmm(X, 3, init_means=[[2.0, 4.5], [1.0, 4.5], [1.5, 4.0]], keep=(0, 1, 3, 10))   # poor start
print(f"EM stopped after {len(fit2['trace'])} iterations; final log-likelihood {fit2['trace'][-1]:.3f}")
print("never decreased:", bool(np.all(np.diff(fit2["trace"]) >= -1e-9)))
sk2 = GaussianMixture(3, covariance_type="full", n_init=10, tol=1e-8, random_state=SEED).fit(X)
print(f"scikit-learn (10 starts): {sk2.score(X) * len(X):.3f}")
order = np.argsort(fit2["mus"][:, 0])
print("our means (sorted by x₁):", fit2["mus"][order].round(2).tolist())
print("sklearn means           :", sk2.means_[np.argsort(sk2.means_[:, 0])].round(2).tolist())
print(f"ARI of EM clusters vs true generating labels: {adjusted_rand_score(c_true, fit2['gamma'].argmax(1)):.2f}")

In [ ]:
# ── 5.5 The EM loop in pictures: ellipses over the iterations ──
fig, axes = plt.subplots(2, 3, figsize=(11, 6.6))
for ax, (t, (p, m, S)) in zip(axes.ravel()[:5], fit2["snaps"].items()):
    g_t, _ = responsibilities(X, p, m, S)
    ax.scatter(*X.T, c=g_t @ rgb, s=10)
    for i in range(3):
        draw_ellipse(ax, m[i], S[i], CLASS_COLORS[i])
        ax.plot(*m[i], "kx")
    ax.set_xlim(-0.5, 5.5); ax.set_ylim(-0.5, 5.5); ax.set_aspect("equal")
    ax.set_title(f"iteration {t}" if t != "final" else f"converged (iteration {len(fit2['trace'])})", fontsize=9)
axes[1, 2].plot(fit2["trace"], "o-", ms=2)
axes[1, 2].set_xlabel("iteration t"); axes[1, 2].set_ylabel("ℓ(θ⁽ᵗ⁾)"); axes[1, 2].set_title("log-likelihood", fontsize=9)
fig.suptitle("EM with full covariances on the SIMULATED three-cluster data (points coloured by responsibilities)", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** From three components bunched in the top-left corner, EM spreads them out: within a few
iterations each ellipse grabs one group, and the ellipses then rotate and stretch to take the group's shape — tight and
tilted, tall and narrow, long and diagonal. The final log-likelihood equals scikit-learn's best of ten starts, and the
EM clusters agree with the generating labels with ARI 0.88 — the disagreements are overlapping points that even the true
model assigns to the "wrong" cluster (Section 3.2: the true model's most probable cluster is right for 96 %). After the first big jump the
log-likelihood creeps up slowly for another ten or so iterations while the components "negotiate" the overlapping
region: EM can be slow, which is why practical implementations need a sensible stopping rule (`tol`).

✏️ **Your turn.** Fit `em_gmm(X, 2, ...)` and `em_gmm(X, 4, ...)`. With $K = 2$, which two groups are merged? With
$K = 4$, does the extra component find anything real, or split a group? (Section 8 asks how to decide.)

### 📘 Lecture notes — learning with partially observed data — the EM algorithm

**Data** $\mathbf x^1,\dots,\mathbf x^N$ — the labels $c^n$ are **not** observed. **Goal:** learn $\pi_i,\boldsymbol\mu_i,\Sigma_i$.

**Optimisation criterion.** We would like to maximise the data log-likelihood
$$\log\prod_{n=1}^{N}p(\mathbf x^n)=\sum_{n}\log\sum_{i=1}^{K}p(\mathbf x^n,c^n=i)=\sum_n\log\sum_i p(\mathbf x^n\mid c^n=i)\,p(c^n=i),$$
which is difficult because $c^n$ is missing (log of a sum: no separation, no closed form). EM instead maximises the
*expected* complete-data log-likelihood $\mathbb E_{p(c^n\mid\mathbf x^n,\theta^{(t)})}\big[\log\prod_n p(\mathbf x^n,c^n)\big]$.

**Procedure.** Guess $\theta^{(0)}$ (e.g. centroids and covariances of the $K$ clusters), then repeat:
- **E-step** (assign clusters): $\gamma_{ni}=P(c^n=i\mid\mathbf x^n)=\dfrac{p(\mathbf x^n\mid c^n=i)\,p(c^n=i)}{\sum_{j}p(\mathbf x^n\mid c^n=j)\,p(c^n=j)}$
  with the current parameters — an $N\times K$ table of responsibilities.
- **M-step** (estimate parameters): $\pi_i=\dfrac{\sum_n\gamma_{ni}}{N}$, $\ \boldsymbol\mu_i=\dfrac{\sum_n\mathbf x^n\gamma_{ni}}{\sum_n\gamma_{ni}}$,
  $\ \Sigma_i=\dfrac{\sum_n(\mathbf x^n-\boldsymbol\mu_i)(\mathbf x^n-\boldsymbol\mu_i)^\top\gamma_{ni}}{\sum_n\gamma_{ni}}$ — the
  fully-observed formulas with $I(c^n=i)$ replaced by $\gamma_{ni}$.

*In words:* inference (E) and weighted maximum likelihood (M) take turns; each solves the other's missing piece.

**Convergence.** The observed-data log-likelihood never decreases (the E-step makes a Jensen lower bound touch $\ell$,
the M-step pushes the bound up). EM converges to a **local** optimum of a non-convex objective, so the answer depends
on the start.

> 🎓 **Exam tip.** The most common exam error is to claim that EM finds the maximum-likelihood estimate. It finds a stationary point
    (usually a local maximum) and only increases — never decreases — the observed-data log-likelihood. Be able to write the
    E- and M-step formulas for a GMM from memory: I(cⁿ = i) → γₙᵢ, one pass → iterations of E and M.

---
## 6 · k-means is the hard-assignment limit of EM

> 📍 **Where we are.** Put k-means and EM side by side. k-means alternates "find the cluster means (centres)" and
> "re-assign samples to clusters" until convergence; clustering with a mixture model alternates "find the cluster means
> *and variances*" (M-step) and "probabilistic cluster assignment" (E-step). k-means makes a **hard** assignment and
> weights each sample with 0 or 1; the mixture makes a **soft** assignment and weights each sample with probabilities.
> Here we show that k-means is not just *similar* to EM — it is a limiting case of it.

Take a GMM with **equal** mixing proportions $\pi_i = 1/K$ and the **same spherical** covariance $\Sigma_i = \sigma^2 I$ for
every cluster, with $\sigma$ fixed. The Gaussian becomes
$\mathcal N(\mathbf x\mid\boldsymbol\mu_i,\sigma^2I)\propto\exp\!\big(-\lVert\mathbf x-\boldsymbol\mu_i\rVert^2/2\sigma^2\big)$, and
the E-step reads

$$\gamma_{ni}=\frac{\exp\!\big(-\lVert\mathbf x^n-\boldsymbol\mu_i\rVert^2/2\sigma^2\big)}{\sum_j\exp\!\big(-\lVert\mathbf x^n-\boldsymbol\mu_j\rVert^2/2\sigma^2\big)} .$$

*In words:* the responsibility depends only on the squared distances to the centres — closer centres get more weight.
Now let $\sigma\to0$. Divide top and bottom by the term of the **nearest** centre: every other term becomes
$\exp\!\big(-(\text{extra squared distance})/2\sigma^2\big)\to0$. So $\gamma_{ni}\to1$ for the nearest centre and 0 for all
others — **k-means' hard assignment**. And with 0/1 weights the M-step for the means,
$\boldsymbol\mu_i=\sum_n\gamma_{ni}\mathbf x^n/\sum_n\gamma_{ni}$, is exactly k-means' "mean of the members". (One can also show
that $-2\sigma^2\ell$ tends to the k-means objective as $\sigma\to0$: the extra terms, of order $\sigma^2\log\sigma^2$,
vanish.) In short: **k-means = EM for a GMM with equal
weights and shared covariance $\sigma^2 I$, in the limit $\sigma\to0$.**

In [ ]:
# ── 6.1 EM with a fixed shared spherical covariance σ²I, for shrinking σ ──
def em_spherical_fixed(X, init_means, sigma, n_iter=200):
    """EM that updates only the means of a GMM with equal weights and covariance σ²I (σ fixed)."""
    mus = np.array(init_means, float)
    for _ in range(n_iter):
        logits = -((X[:, None, :] - mus[None]) ** 2).sum(axis=2) / (2 * sigma ** 2)
        gamma = np.exp(logits - logsumexp(logits, axis=1)[:, None])           # E-step
        new = gamma.T @ X / gamma.sum(axis=0)[:, None]                        # M-step (means only)
        if np.allclose(new, mus, atol=1e-10):
            break
        mus = new
    return mus, gamma


init = X[[np.flatnonzero(c_true == i)[0] for i in range(3)]]   # same start for all runs: first sample of each group
                                                              # (chosen this way only so that colours match Section 1)
lab_k, cen_k = kmeans_hard(X, 3, init=init)
rows = []
for sigma in [2.0, 1.0, 0.5, 0.25, 0.1, 0.03]:
    mus_s, gam_s = em_spherical_fixed(X, init, sigma)
    rows.append({"σ": sigma, "mean of max_i γ": gam_s.max(1).mean(), "samples with max γ > 0.99": np.mean(gam_s.max(1) > 0.99),
                 "max |μ_EM − μ_k-means|": np.abs(mus_s - cen_k).max(),        # component i started where centre i did
                 "same labels as k-means": np.mean(gam_s.argmax(1) == lab_k)})
print(pd.DataFrame(rows).round(3).to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
for ax, sigma in zip(axes[:2], [1.0, 0.1]):
    mus_s, gam_s = em_spherical_fixed(X, init, sigma)
    ax.scatter(*X.T, c=gam_s @ rgb, s=14); ax.scatter(*mus_s.T, marker="X", s=100, c="k")
    ax.set_title(f"σ = {sigma}: colours = responsibilities"); ax.set_xlabel("$x_1$"); ax.set_aspect("equal")
axes[0].set_ylabel("$x_2$")
sig_grid = np.geomspace(0.02, 3, 25)
axes[2].semilogx(sig_grid, [em_spherical_fixed(X, init, s)[1].max(1).mean() for s in sig_grid], "o-", ms=3)
axes[2].axhline(1 / 3, color="0.6", ls=":"); axes[2].invert_xaxis()
axes[2].set_xlabel("σ (decreasing →)"); axes[2].set_ylabel("average max responsibility")
axes[2].set_title("soft → hard as σ → 0")
plt.tight_layout(); plt.show()

🔍 **What just happened?** With a large $\sigma$ every centre is "close" to every point: above $\sigma\approx1.3$ the
three means collapse onto the overall mean and every responsibility is exactly 1/3 (the flat start of the right panel).
As $\sigma$ shrinks the responsibilities harden: at $\sigma = 1$ the most probable cluster already agrees with k-means for
every sample, although no sample is yet assigned with probability above 0.99; at $\sigma = 0.1$, 99 % of the samples are,
and the EM means coincide with the k-means centres to three decimals; at $\sigma = 0.03$ the two algorithms are identical. The right panel shows the smooth transition from
soft to hard. This is the precise sense in which a mixture model is "the probabilistic version of k-means" —
and it shows what k-means implicitly assumes: **round clusters of equal size and equal spread**. Our three clusters
violate all three assumptions, which is why the full-covariance GMM of Section 5 describes them better.

✏️ **Your turn.** Start both algorithms from three samples of the *same* group, e.g. `init = X[np.flatnonzero(c_true == 0)[:3]]`.
Do EM (small σ) and k-means still agree with each other? Do they still find the three groups? (The limit holds for any
start — including starts that lead both into the same poor local optimum.)

### 📘 Lecture notes — k-means versus mixture models

| | k-means | Gaussian mixture (EM) |
|---|---|---|
| step 1 | re-assign each sample to the nearest centre | E-step: probabilistic assignment $\gamma_{ni}$ |
| step 2 | find the cluster means | M-step: cluster means, covariances and sizes |
| weight of a sample in cluster $i$ | 0 or 1 (hard) | a probability (soft) |

**The limit.** For a GMM with equal $\pi_i=1/K$ and a shared, fixed covariance $\sigma^2I$,
$\gamma_{ni}\propto\exp(-\lVert\mathbf x^n-\boldsymbol\mu_i\rVert^2/2\sigma^2)$. As $\sigma\to0$ the nearest centre gets
$\gamma_{ni}\to1$ and all others $\to0$, and the M-step for the means becomes "mean of the members": EM becomes
k-means. *In words:* k-means is the hard-assignment, round-cluster special case of a mixture model — which is why
mixture models are called the probabilistic version of k-means.

> 🎓 **Exam tip.** Be ready to name the GMM assumptions under which EM reduces to k-means: equal πᵢ, Σᵢ = σ²I shared and fixed, σ → 0.
    Consequence: k-means prefers spherical clusters of similar size.

---
## 7 · EM in practice: restarts and degenerate solutions

> 📍 **Where we are.** Section 5 promised a stationary point, not the best one, and warned about unbounded likelihoods.
> Here is the practical recipe, and we watch both problems happen — the second one on expression data.

**EM in practice.** Initialise the parameters (for example to random values); results can vary with the initialisation;
so run EM from **multiple** starting points — several random initialisations, or the result of a simpler method such as
**k-means** — and **select the result with the highest data log-likelihood**.

### 7.1 Many starts, many answers

In [ ]:
# ── 7.1 Thirty random starts on five groups (SIMULATED) ──
rng = np.random.default_rng(SEED + 71)
centres5 = np.array([[0, 0], [0, 3.0], [3.0, 0], [3.0, 3.0], [6.5, 1.5]])
X5 = np.vstack([rng.multivariate_normal(m, [[0.3, 0.1], [0.1, 0.3]] if i % 2 else [[0.25, -0.05], [-0.05, 0.35]], 100)
                for i, m in enumerate(centres5)])
fits5 = [em_gmm(X5, 5, rng=np.random.default_rng(100 + s)) for s in range(30)]
ll5 = np.array([f["trace"][-1] for f in fits5])
_, cen5 = kmeans_hard(X5, 5, n_starts=10)
fit_km = em_gmm(X5, 5, init_means=cen5)                                       # k-means initialisation
print("final log-likelihoods of 30 random starts (sorted):\n", np.sort(ll5).round(1))
print(f"{np.mean(ll5 > ll5.max() - 0.01):.0%} of random starts reach the best value {ll5.max():.1f}; "
      f"k-means-initialised EM reaches {fit_km['trace'][-1]:.1f}")
print("every run was monotone:", all(np.all(np.diff(f["trace"]) >= -1e-8) for f in fits5))

fig, axes = plt.subplots(1, 3, figsize=(11, 3.3))
axes[0].plot(np.sort(ll5), "o", ms=4)
axes[0].set_xlabel("run (sorted)"); axes[0].set_ylabel("final log-likelihood"); axes[0].set_title("30 random starts")
for ax, f, title in [(axes[1], fits5[int(np.argmin(ll5))], "worst start"), (axes[2], fits5[int(np.argmax(ll5))], "best start")]:
    ax.scatter(*X5.T, s=4, color="0.6")
    for i in range(5):
        draw_ellipse(ax, f["mus"][i], f["Sigmas"][i], CLASS_COLORS[i % len(CLASS_COLORS)])
    ax.set_title(f"{title}: ℓ = {f['trace'][-1]:.1f} (SIMULATED)"); ax.set_aspect("equal")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Every run increased its likelihood monotonically — and yet they ended at many different
values. Only 53 % of the random starts found the best solution (ℓ = −1536.3); in the worst (ℓ = −1651.1) one wide
component covers both lower groups while another has degenerated into a thin sliver stretched across the gap between two
groups — a local maximum that EM cannot escape, because any small change lowers the likelihood. Starting from k-means
centres gave the best solution here in a single run. The recipe — several starts, keep the highest log-likelihood
(scikit-learn: `n_init`, `init_params="kmeans"`) — is not optional.

### 7.2 Degenerate solutions: when the likelihood goes to infinity

Put one component's mean exactly on a data point, $\mu_1 = x^1$, and shrink its standard deviation. That point's density
under component 1 is

$$\mathcal N(x^1\mid\mu_1=x^1,\sigma_1^2)=\frac{1}{\sqrt{2\pi}\,\sigma_1}\;\xrightarrow{\;\sigma_1\to0\;}\;\infty ,$$

while the other component keeps every other point's density finite. *In words:* the likelihood can be made as large as
we like by letting one component "collapse" onto a single point. The maximum-likelihood estimate of an unconstrained GMM
**does not exist**, and EM — which only ever climbs — can climb into such a hole. Does that happen with expression data?

**Expression data with a detection floor.** In the standard preprocessing of the leukaemia data of Golub *et al.*
(Dudoit, Fridlyand & Speed 2002), intensities below 100 were set to 100 before taking $\log_{10}$, so every gene has a pile of values at
exactly $\log_{10}100 = 2$. The simulated leukaemia data imitate this floor. We take the most variable gene with 15–30
such floored values and fit a two-component 1-D mixture to its 72 values, first with no constraint, then with a
standard-deviation floor of 0.05 (a plausible minimum measurement spread on this scale).

In [ ]:
# ── 7.2 A Gaussian mixture collapses onto the detection floor of (SIMULATED) expression data ──
leuk = simulate_leukemia_expression()                         # the course's standard simulated data set (fixed seed)
print("source:", leuk["source"])
Xg = leuk["X"]
n_floor = np.isclose(Xg, 2.0).sum(axis=0)
print(f"{np.isclose(Xg, 2.0).mean():.1%} of all expression values sit exactly at the floor log10(100) = 2")
cand = np.flatnonzero((n_floor >= 15) & (n_floor <= 30))
j = cand[np.argmax(Xg[:, cand].var(axis=0))]
x_g = Xg[:, j]
print(f"gene {leuk['genes'][j]}: {n_floor[j]} of 72 values at the floor")

free = em_gmm_1d(x_g, mus=[2.2, 3.0], sds=[0.5, 0.5], pis=[0.5, 0.5], n_iter=300)
floored = em_gmm_1d(x_g, mus=[2.2, 3.0], sds=[0.5, 0.5], pis=[0.5, 0.5], n_iter=300, sd_floor=0.05)
n_free = len(free["trace"])
for t in [t for t in [0, 10, 20, 30, 40, 45, 50] if t < n_free - 2] + [n_free - 2, n_free - 1]:
    print(f"  unconstrained, iteration {t:2d}: ℓ = {free['trace'][t]:8.2f}, σ = {free['sd_trace'][t].round(5)}")
print(f"with σ ≥ 0.05: converged at ℓ = {floored['trace'][-1]:.2f}, μ = {floored['mus'].round(2)}, "
      f"σ = {floored['sds'].round(3)}, π = {floored['pis'].round(2)}")

fin = np.flatnonzero(np.isfinite(free["trace"]))
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
axes[0].hist(x_g, bins=30, color="0.8", edgecolor="0.5")
axes[0].set_xlabel("log10 expression"); axes[0].set_ylabel("patients")
axes[0].set_title(f"SIMULATED gene {leuk['genes'][j]}: {n_floor[j]} values piled at 2.0")
axes[1].semilogy(fin, free["sd_trace"][fin, 0], "o-", ms=3, label="σ₁ (unconstrained)")
axes[1].semilogy(np.arange(len(floored["sd_trace"])), floored["sd_trace"][:, 0], "-", label="σ₁ with floor 0.05")
axes[1].set_xlabel("iteration"); axes[1].set_ylabel("σ₁ (log scale)"); axes[1].legend(fontsize=7)
axes[1].set_title("one component shrinks onto the floor")
axes[2].plot(fin, free["trace"][fin], "o-", ms=3, label="unconstrained (then ℓ → ∞)")
axes[2].plot(floored["trace"], "-", label="σ ≥ 0.05")
axes[2].set_xlabel("iteration"); axes[2].set_ylabel("log-likelihood ℓ"); axes[2].legend(fontsize=7)
axes[2].set_title("EM keeps climbing — to infinity")
plt.tight_layout(); plt.show()

🔍 **What just happened?** On this gene, unconstrained EM did exactly what it promises — the log-likelihood never
decreased — and drove itself into a degenerate solution. For about forty iterations it looked harmless: component 1
narrowed slowly around the low values while ℓ crept up by a few hundredths per iteration (a careless stopping rule would
have declared convergence here). Then component 1 locked onto the pile of floored values: σ₁ fell from about 0.1 to
below 0.001 within about fifteen iterations, ℓ jumped from about −40 to +45, and one step later σ₁ was 0 to machine
precision, ℓ became undefined (`nan`) and the loop stopped. With a floor of 0.05 on σ the fit converges to a finite
solution in which one component still sits on the floor value 2.0 with the smallest allowed width, and the other
describes the expressed values.

What did the mixture "discover"? Not a biological population, but a **preprocessing artefact** — the thresholding of
low intensities. The fixes, in increasing order of thought: a **variance floor / regulariser** (scikit-learn's
`reg_covar` adds a small constant to every variance, like our `reg` in `m_step`); a **prior** on the variances, turning
MLE into MAP ([Module 05](05_likelihood_mle_map.ipynb)); or a model that knows the values below 100 are **censored**
("somewhere below 100") rather than exactly equal.

✏️ **Your turn.** Fit the gene with `sd_floor=0.2` and with three components (add a third mean, e.g. `mus=[2.2, 3.0, 3.6]`).
Which fit would you report, and what would you say about the pile at 2.0?

### 📘 Lecture notes — EM in practice

**Recipe.**
1. Initialise the parameters — random values, randomly chosen samples as means, or the result of a simpler method
   such as k-means.
2. Run EM to convergence. Results can vary with the initialisation (EM finds a local optimum).
3. Repeat from several different initialisations (multiple random starts and/or k-means starts).
4. Keep the run with the **highest observed-data log-likelihood**
   $\sum_n\log\sum_i\pi_i\,\mathcal N(\mathbf x^n\mid\boldsymbol\mu_i,\Sigma_i)$.

**Caveat — degenerate solutions.** Step 4 is only meaningful when the likelihood is bounded. A Gaussian component whose
mean sits on one data point (or on a pile of identical values, such as a detection floor) and whose variance shrinks
has density $1/(\sqrt{2\pi}\sigma)\to\infty$, so the "best" likelihood would always be a collapsed component. Guard
against it with a variance floor (`reg_covar`), a prior on the variances (MAP), or a censored-data model.

> 🎓 **Exam tip.** "Select the highest likelihood" is only meaningful when the likelihood is bounded: with free variances, a
    collapsed component would always win. Mention variance floors / regularisation whenever you describe fitting GMMs.

---
## 8 · How many clusters?

> 📍 **Where we are.** Like k-means, a mixture needs $K$. Unlike k-means, it has a likelihood, which gives principled
> ways to choose $K$. The general theory of model selection is [Module 23](23_model_selection.ipynb).

The training log-likelihood is useless for choosing $K$: a model with more components can always fit the training data
at least as well. Two remedies:

- **Held-out likelihood (cross-validation).** Fit on training data, then compute the log-likelihood of
  *validation* data the model has not seen, and pick the number of clusters whose model is most accurate on them. Too
  few components underfit; too many fit noise and generalise worse. Because the validation labels are unknown, the score
  must be the **observed-data** log-likelihood
  $\log p(\mathbf x^1,\dots,\mathbf x^N)=\sum_n\log\sum_i\pi_i\,\mathcal N(\mathbf x^n\mid\boldsymbol\mu_i,\Sigma_i)$ (it is
  sometimes written loosely as $\sum_n\log p(\mathbf x^n\mid c^n)p(c^n)$, which would require knowing $c^n$ — read it
  as summed over $c^n$).
- **BIC (Bayesian information criterion; Schwarz 1978):**
  $$\mathrm{BIC}(K) = -2\,\ell(\hat\theta_K) + p(K)\,\log N .$$
  *In words:* reward the fit (higher log-likelihood → lower BIC) but charge $\log N$ for each of the $p(K)$ free parameters
  (Section 2.4); pick the $K$ with the **lowest** BIC.

In [ ]:
# ── 8.1 Held-out log-likelihood and BIC for K = 1..7 (SIMULATED from the three-cluster model) ──
rng = np.random.default_rng(SEED + 81)
X_train, c_train = sample_gmm(rng, 300, PI_TRUE, MU_TRUE, SIGMA_TRUE)
X_valid, _ = sample_gmm(rng, 300, PI_TRUE, MU_TRUE, SIGMA_TRUE)
Ks = range(1, 8)
res, fits_by_K = [], {}
for K in Ks:
    f = max((em_gmm(X_train, K, rng=np.random.default_rng(10 * K + s)) for s in range(5)), key=lambda f: f["trace"][-1])
    fits_by_K[K] = f
    _, ll_valid = responsibilities(X_valid, f["pis"], f["mus"], f["Sigmas"])
    bic = -2 * f["trace"][-1] + n_params_gmm(K, 2) * np.log(len(X_train))
    res.append({"K": K, "train ℓ / N": f["trace"][-1] / 300, "validation ℓ / N": ll_valid / 300, "BIC": bic})
res = pd.DataFrame(res).set_index("K")
print(res.round(3))
gm3 = GaussianMixture(3, n_init=5, random_state=SEED).fit(X_train)
print("our BIC formula reproduces scikit-learn's:", np.isclose(gm3.bic(X_train),
      -2 * gm3.score(X_train) * len(X_train) + n_params_gmm(3, 2) * np.log(len(X_train))))

fig, axes = plt.subplots(2, 3, figsize=(11, 6.4))
for ax, col in zip(axes[0], res.columns):
    ax.plot(list(Ks), res[col], "o-")
    ax.axvline(res[col].idxmin() if col == "BIC" else res[col].idxmax(), color="tab:red", ls=":")
    ax.set_xlabel("number of components K"); ax.set_title(col)
for ax, K in zip(axes[1], [1, 3, 6]):                              # the fitted models: too simple, right, too complex
    ax.scatter(*X_valid.T, s=4, color="0.6")
    for i in range(K):
        draw_ellipse(ax, fits_by_K[K]["mus"][i], fits_by_K[K]["Sigmas"][i], "tab:red")
    ax.set_title(f"K = {K}: validation ℓ/N = {res.loc[K, 'validation ℓ / N']:.3f}", fontsize=9)
    ax.set_xlim(-0.5, 5.5); ax.set_ylim(-0.5, 5.5); ax.set_aspect("equal")
fig.suptitle("choosing K on SIMULATED data with 3 true clusters (top, dotted: best K by each criterion;\n"
             "bottom: fitted 2-sd ellipses over the validation data)", fontsize=10)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The training log-likelihood keeps rising with $K$ (left) — useless for choosing. The
validation log-likelihood (middle) rises up to $K = 3$ and then slowly declines: extra components buy nothing on new
data. BIC (right) is lowest at $K = 3$, the true number — but only just below $K = 2$ (1575 vs 1580): two of our clusters
overlap, so the evidence for the third one is real but modest. Our parameter count gives exactly
scikit-learn's BIC. The bottom row shows what the numbers mean: one ellipse cannot describe three groups ($K = 1$),
three ellipses sit on the three groups ($K = 3$), and six ellipses start splitting groups into pieces — one of them a
thin sliver — that describe the training sample's noise rather than structure ($K = 6$). On real data the answers are rarely this crisp, and BIC counts
the clusters the *model and features* can resolve — if the true clusters are not Gaussian, BIC tends to add components to
describe their shape.

### A different answer: let the data decide

A **nonparametric Bayesian** alternative avoids choosing $K$ at all: put a **Dirichlet-process prior** on the mixing proportions,
i.e. assume an *infinite* number of clusters a priori, of which only a finite number are used to model any finite data
set. Its most intuitive description is the **Chinese restaurant process**: customers (samples) enter one by one; customer
$n+1$ sits at an occupied table with probability proportional to the number of people already there, or opens a **new**
table with probability proportional to a concentration parameter $\alpha$:

$$P(\text{new table}) = \frac{\alpha}{n+\alpha}, \qquad P(\text{table } k) = \frac{n_k}{n+\alpha}.$$

*In words:* popular clusters attract new members ("the rich get richer"), but there is always some chance of a new
cluster, so the number of clusters grows slowly with the data — and is learned rather than fixed. Fitting such models
needs the sampling methods of [Module 21](21_bayesian_learning_mcmc.ipynb); here we just simulate the prior.

In [ ]:
# ── 8.2 The Chinese restaurant process: how many tables (clusters) for n customers? ──
def crp_tables(n: int, alpha: float, rng: np.random.Generator) -> np.ndarray:
    """Seat n customers by the Chinese restaurant process; return the number of occupied tables after each customer."""
    counts, n_tables = [], np.zeros(n, int)
    for i in range(n):
        p = np.array(counts + [alpha], float) / (i + alpha)          # existing tables ∝ size, new table ∝ α
        k = rng.choice(len(p), p=p)
        if k == len(counts):
            counts.append(1)
        else:
            counts[k] += 1
        n_tables[i] = len(counts)
    return n_tables


rng = np.random.default_rng(SEED + 82)
n_cust = np.arange(1, 1001)
fig, ax = plt.subplots(figsize=(6.5, 3.3))
for alpha, col in zip([0.5, 2.0, 10.0], ["tab:blue", "tab:orange", "tab:green"]):
    for rep in range(3):
        ax.plot(n_cust, crp_tables(1000, alpha, rng), color=col, lw=0.8, alpha=0.7, label=f"α = {alpha}" if rep == 0 else None)
    expected = np.cumsum(alpha / (alpha + n_cust - 1))              # E[#tables] = Σᵢ α / (α + i − 1)
    ax.plot(n_cust, expected, "k--", lw=1)
ax.set_xscale("log"); ax.set_xlabel("number of customers (samples)"); ax.set_ylabel("occupied tables (clusters)")
ax.set_title("Chinese restaurant process: clusters grow like α·log n (dashed: expectation)"); ax.legend()
plt.tight_layout(); plt.show()

🔍 **What just happened?** For every $\alpha$ the number of clusters keeps growing, but only logarithmically once the
number of customers is much larger than $\alpha$ (straight lines on the log axis): 1,000 samples use a few tables for $\alpha = 0.5$, about a dozen for $\alpha = 2$ and a few dozen
for $\alpha = 10$. The simulations scatter around the expected value $\sum_{i=1}^{n}\alpha/(\alpha+i-1)$. In a Dirichlet-process
mixture this prior is combined with the likelihood, so the data decide how many of the infinitely many available clusters
are actually used.

✏️ **Your turn.** In cell 8.1 use only 60 training samples (`sample_gmm(rng, 60, ...)`) and re-run. Which $K$ do BIC and
the validation likelihood choose now, and do they still agree? (BIC's penalty grows with $\log N$ but the evidence for
each extra cluster grows with $N$.)

### 📘 Lecture notes — how to select the number of clusters

**Cross-validation.** Fit mixtures with $K = 1, 2, 3, \dots$ components on training data; compute the likelihood of
held-out (validation) data,
$$\log p(\mathbf x^1,\dots,\mathbf x^N)=\sum_{n=1}^{N}\log\sum_{i=1}^{K}p(\mathbf x^n\mid c^n=i)\,p(c^n=i),$$
and choose the $K$ whose model is most accurate on them. *In words:* a model with too few components misses
structure, one with too many fits noise; both predict new data worse. The training likelihood cannot be used — it
never decreases with $K$.

**Nonparametric Bayesian prior.** Place a **Dirichlet-process prior** on the mixture proportions: an *infinite*
mixture model assumes infinitely many clusters a priori, of which only a finite number are used to model any finite
data set. Its sequential description is the **Chinese restaurant process**: customer $n+1$ joins table $k$ with
probability $n_k/(n+\alpha)$ or opens a new table with probability $\alpha/(n+\alpha)$; the expected number of tables
grows like $\alpha\log n$.

> 🎓 **Exam tip.** Explain why the training likelihood cannot choose K and what BIC's penalty p(K) log N does; know that BIC and
    held-out likelihood can disagree, and that a Dirichlet-process mixture learns the number of clusters as part of
    the model.

---
## 9 · Bernoulli and binomial mixtures: population structure from genotypes

> 📍 **Where we are.** Everything so far assumed Gaussian components — the right choice for clustering genes and samples
> by expression levels. Other mixtures are used in biology: **mixtures of Bernoulli distributions for discrete data** —
> clustering genome sequence data to find population structure, or quantifying expression from RNA-seq by assigning
> reads to the genes they map to — and **mixtures of HMMs**, more complex models for clustering genome sequences. Here
> we work out the Bernoulli mixture: it is the model behind the genotype clustering of Section 1.

**The Bernoulli mixture.** Data points $\mathbf x^1,\dots,\mathbf x^N$ have $J$ binary entries (0 or 1; often written
loosely as $\mathbf x^n\in\mathbb R^J$), with latent clusters $c^1,\dots,c^N$. Generative story: $c^n = i$ with probability
$\pi_i$; then every entry is an independent coin toss, $x^n_j\mid c^n=i\sim\mathrm{Bern}(\mu_{ij})$, with $\mu_{ij}$ the
probability of a 1 for variable $j$ in cluster $i$. The distribution is again
$p(\mathbf x^n)=\sum_i p(\mathbf x^n\mid c^n=i)\,p(c^n=i)$ with
$p(\mathbf x^n\mid c^n=i)=\prod_j\mu_{ij}^{x^n_j}(1-\mu_{ij})^{1-x^n_j}$; inference is Bayes' rule as in Section 3; and the
EM M-step is $\pi_i^{\text{new}}=\sum_n\gamma_{ni}/N$ and
$\mu_{ij}^{\text{new}}=\sum_n x^n_j\,\gamma_{ni}/\sum_n\gamma_{ni}$. *In words:* the cluster size is its total
responsibility, and the probability of a 1 in cluster $i$ is the responsibility-weighted fraction of 1s.

**Genotypes are two coin tosses.** A diploid genotype $x_{nj}\in\{0,1,2\}$ counts copies of one allele (Module 06's coding
AA = 0, AT = 1, TT = 2): one toss per chromosome, each landing on the counted allele with that population's allele
frequency $\mu_{ij}$. Under Hardy–Weinberg equilibrium the genotype is **binomial** with 2 trials:

$$P(x_{nj}\mid c^n=i)=\binom{2}{x_{nj}}\mu_{ij}^{\,x_{nj}}(1-\mu_{ij})^{2-x_{nj}},\qquad
\log P(\mathbf x^n\mid c^n=i)=\sum_{j=1}^{J}\Big[x_{nj}\log\mu_{ij}+(2-x_{nj})\log(1-\mu_{ij})\Big]+\text{const}.$$

*In words:* the probability of an individual's genotypes under population $i$ is a product over SNPs (assumed independent
given the population — the naive-Bayes assumption of [Module 08](08_knn_naive_bayes.ipynb)); in logs, a sum. The constant
$\sum_j\log\binom{2}{x_{nj}}$ is the same for every population, so it cancels in the responsibilities.

**EM, adapted to two trials:**

- **E-step:** $\gamma_{ni}\propto\pi_i\,P(\mathbf x^n\mid c^n=i)$, computed in log space;
- **M-step:** $\pi_i=\frac1N\sum_n\gamma_{ni}$ and
  $$\mu_{ij}=\frac{\sum_n\gamma_{ni}\,x_{nj}}{2\sum_n\gamma_{ni}} .$$

*In words:* the allele frequency of SNP $j$ in population $i$ is the responsibility-weighted count of the allele divided by
the responsibility-weighted number of chromosomes (2 per person). For a Bernoulli mixture (one toss) the 2 disappears and
we are back at the M-step above.

⚠️ **A pseudo-count.** If an allele is never seen among a cluster's members, $\mu_{ij}=0$ and $\log\mu_{ij}=-\infty$ — a
degenerate solution again. We add $a = 1$ pseudo-allele of each kind: $\mu_{ij}=(\sum_n\gamma_{ni}x_{nj}+a)/(2\sum_n\gamma_{ni}+2a)$.
This is MAP estimation with a Beta prior ([Module 05](05_likelihood_mle_map.ipynb)); EM then never decreases the
log-likelihood **plus** log-prior — the objective it actually optimises — which is what we monitor.

In [ ]:
# ── 9.1 Genotypes of three populations (SIMULATED with the Balding–Nichols model) ──
rng = np.random.default_rng(SEED + 91)
G, pop, freqs = simulate_genotypes(rng, n_samples=300, n_snps=500, n_pops=3, fst=0.05)
G = G.astype(float)
print("genotype matrix:", G.shape, "| individuals per population:", np.bincount(pop))
print("genotype counts (0/1/2):", np.bincount(G.astype(int).ravel()))
print("mean |frequency difference| between populations 0 and 1:", np.abs(freqs[0] - freqs[1]).mean().round(3))

In [ ]:
# ── 9.2 EM for a binomial mixture, from scratch ──
def em_binomial_mixture(G, K, n_iter=300, a=1.0, tol=1e-6, rng=None):
    """EM (MAP with a pseudo-count a) for a mixture of per-SNP binomial(2, μ_ij) distributions.

    Returns dict with pis, mus (K × J allele frequencies), gamma (N × K) and the trace of log-likelihood + log-prior.
    """
    rng = rng if rng is not None else np.random.default_rng(0)
    N, J = G.shape
    pis, mus = np.full(K, 1.0 / K), rng.uniform(0.2, 0.8, size=(K, J))
    const = np.log(np.where(G == 1, 2.0, 1.0)).sum(axis=1)                   # Σ_j log C(2, x_nj)
    trace = []
    for t in range(n_iter):
        log_joint = np.log(np.maximum(pis, 1e-300)) + G @ np.log(mus).T + (2 - G) @ np.log(1 - mus).T + const[:, None]
        log_px = logsumexp(log_joint, axis=1)
        trace.append(log_px.sum() + a * (np.log(mus) + np.log(1 - mus)).sum())    # objective EM maximises
        if t > 0 and abs(trace[-1] - trace[-2]) < tol:
            break
        gamma = np.exp(log_joint - log_px[:, None])                             # E-step
        Nk = gamma.sum(axis=0)                                                  # M-step
        pis = Nk / N
        mus = (gamma.T @ G + a) / (2 * Nk[:, None] + 2 * a)
    return dict(pis=pis, mus=mus, gamma=gamma, trace=np.array(trace))


t0 = time.perf_counter()
fits_g = [em_binomial_mixture(G, 3, rng=np.random.default_rng(s)) for s in range(5)]
fit_g = max(fits_g, key=lambda f: f["trace"][-1])
print(f"5 starts in {time.perf_counter() - t0:.2f} s; best objective {fit_g['trace'][-1]:.1f} after {len(fit_g['trace'])} iterations")
print("objective never decreased:", all(np.all(np.diff(f["trace"]) >= -1e-8) for f in fits_g))
lab_g = fit_g["gamma"].argmax(axis=1)
print(f"ARI with the true populations: {adjusted_rand_score(pop, lab_g):.2f}; "
      f"individuals with max responsibility < 0.95: {np.sum(fit_g['gamma'].max(1) < 0.95)}")
print(pd.crosstab(lab_g, pop, rownames=["cluster"], colnames=["true population"]))
match = [np.bincount(pop[lab_g == k], minlength=3).argmax() for k in range(3)]      # cluster → population
r = np.corrcoef(fit_g["mus"].ravel(), freqs[match].ravel())[0, 1]
print(f"correlation between estimated and true allele frequencies: {r:.3f}")

In [ ]:
# ── 9.3 Pictures: genotypes sorted by cluster, frequencies recovered, objective ──
order = np.lexsort((pop, lab_g))
fig, axes = plt.subplots(1, 3, figsize=(12, 3.3), gridspec_kw=dict(width_ratios=[1.4, 1, 1]))
axes[0].imshow(G[order][:, :120], aspect="auto", cmap="Greys", interpolation="nearest")
for b in np.flatnonzero(np.diff(lab_g[order])) + 0.5:
    axes[0].axhline(b, color="tab:red", lw=1.5)
axes[0].set_xlabel("first 120 SNPs"); axes[0].set_ylabel("individuals (sorted by EM cluster)")
axes[0].set_title("genotypes 0/1/2 (SIMULATED), red = cluster borders", fontsize=9)
axes[1].scatter(freqs[match].ravel(), fit_g["mus"].ravel(), s=3, alpha=0.4)
axes[1].plot([0, 1], [0, 1], "k--", lw=1)
axes[1].set_xlabel("true allele frequency"); axes[1].set_ylabel("EM estimate μᵢⱼ"); axes[1].set_title(f"r = {r:.3f}", fontsize=9)
axes[2].plot(fit_g["trace"], "o-", ms=3)
axes[2].set_xlabel("iteration"); axes[2].set_ylabel("log-likelihood + log-prior"); axes[2].set_title("EM objective", fontsize=9)
plt.tight_layout(); plt.show()

🔍 **What just happened?** From 500 SNPs alone, EM sorted 300 simulated individuals into three clusters that match the
three populations perfectly, with every individual assigned with probability above 0.95. The recovered allele frequencies track
the true ones (the scatter around the diagonal is binomial sampling noise from ~100 people per population), and the
objective rose monotonically in a handful of iterations. With $F_{ST} = 0.05$ each SNP differs only a little between
populations — the mean absolute frequency difference printed above is small — but hundreds of SNPs add up, which is
why genome-wide data can resolve ancestry even when no single marker can.

How much signal is needed? Below, $F_{ST}$ (how different the populations are) and $J$ (how many SNPs) are varied.

In [ ]:
# ── 9.4 When can populations be separated? ARI over F_ST × number of SNPs (SIMULATED) ──
grid_res = {}
for fst in [0.005, 0.01, 0.02, 0.05]:
    row = {}
    for n_snps in [100, 500, 2000]:
        Gs, pops, _ = simulate_genotypes(np.random.default_rng(int(1e4 * fst) + n_snps), 300, n_snps, n_pops=3, fst=fst)
        best = max((em_binomial_mixture(Gs.astype(float), 3, rng=np.random.default_rng(s)) for s in range(3)),
                   key=lambda f: f["trace"][-1])
        row[f"{n_snps} SNPs"] = adjusted_rand_score(pops, best["gamma"].argmax(1))
    grid_res[f"F_ST = {fst}"] = row
print("ARI between EM clusters and true populations (300 individuals):\n", pd.DataFrame(grid_res).T.round(2))

🔍 **What just happened?** Separation depends on both differentiation and data: with $F_{ST} = 0.05$, 100 SNPs already
give ARI 0.90 and 500 a perfect split; with $F_{ST} = 0.02$, 500 SNPs are needed; with $F_{ST} = 0.01$, 2,000; and with
$F_{ST} = 0.005$ — populations that are genetically almost identical — EM finds essentially nothing even with 2,000 SNPs
(the 0.17 at 500 SNPs is a weak, partial structure). Here 300 individuals were used; more individuals also help. Human populations from different continents
differ by an $F_{ST}$ of very roughly 0.1, neighbouring populations within a continent by far less, which is why studies
of fine-scale structure need hundreds of thousands of SNPs (Module 14's PCA of European genomes is the classic example).

**From clusters to admixture.** Our model gives each person **one** population. STRUCTURE and ADMIXTURE let each
individual have **admixture proportions** — a fraction of their genome from each population — and fit the model with
Gibbs sampling ([Module 21](21_bayesian_learning_mcmc.ipynb)) or fast optimisation. Our soft responsibilities are *not*
admixture proportions: a responsibility of 0.5 means "uncertain which population", not "half of each".

✏️ **Your turn.** In 9.1 set `fst=0.01` and `n_snps=2000`, re-run 9.1–9.3, and look at the responsibilities: are the
mistakes concentrated in individuals with uncertain responsibilities?

### 📘 Lecture notes — other mixtures — Bernoulli mixture models

**Which mixture?** Gaussian for continuous measurements; **Bernoulli** for binary/discrete data (genotypes →
population structure; RNA-seq reads → genes); **mixtures of HMMs** for sequences.

**Bernoulli mixture in four lines.** Generation: $c^n=i$ with probability $\pi_i$, then each $x^n_j$ is a coin toss
with probability $\mu_{ij}$. Distribution: $p(\mathbf x^n)=\sum_i\pi_i\prod_j\mu_{ij}^{x^n_j}(1-\mu_{ij})^{1-x^n_j}$.
Inference: $\gamma_{ni}=p(c^n=i\mid\mathbf x^n)$ by Bayes' rule. EM: E-step computes $\gamma_{ni}$; M-step
$\pi_i=\sum_n\gamma_{ni}/N$, $\mu_{ij}=\sum_n x^n_j\gamma_{ni}/\sum_n\gamma_{ni}$.

**Genotypes** (0/1/2 copies of an allele) are two coin tosses per locus: a binomial with two trials, so the M-step
divides by $2\sum_n\gamma_{ni}$; a pseudo-count (Beta prior) keeps every frequency away from exactly 0 or 1.

> 🎓 **Exam tip.** Be able to write the E- and M-step of a Bernoulli mixture and adapt them to a binomial with two trials (divide by
    2 Σₙ γₙᵢ). Remember that responsibilities are not admixture proportions: 0.5 means "unsure which population", not
    "half of each".

---
## 10 · A mixture model of leukaemia samples

> 📍 **Where we are.** Golub *et al.* (1999) showed that clustering expression profiles can **discover** cancer classes
> without being told the diagnosis: their self-organising map split the samples into two groups that largely matched
> ALL and AML. Can a mixture model do it too — even find the B- and T-lineage subtypes of ALL — and does it add honest
> uncertainty about borderline patients? Our SIMULATED data come with every patient's true class and subtype.

Section 2.4 showed that a full-covariance GMM on many genes is impossible with 72 patients. We use the 500 most
variable genes (centred, as in Module 12) and a **diagonal** covariance: within a cluster, genes are modelled as
independent, each with its own variance — the Gaussian version of naive Bayes ([Module 08](08_knn_naive_bayes.ipynb)).
The E-step log-likelihood is a sum over genes,

$$\log\mathcal N(\mathbf x\mid\boldsymbol\mu_i,\mathrm{diag}(\sigma_{i1}^2,\dots,\sigma_{iJ}^2))=-\frac12\sum_{j=1}^{J}\Big[\log(2\pi\sigma_{ij}^2)+\frac{(x_j-\mu_{ij})^2}{\sigma_{ij}^2}\Big].$$

*In words:* each gene contributes its own one-dimensional Gaussian log-density. A variance floor is again a necessary
safety net: a gene that sat at the detection floor in **every** member of a cluster would get $\sigma_{ij}=0$ and an
infinite likelihood (Section 7.2).

In [ ]:
# ── 10.1 Diagonal-covariance GMM with K = 2 on 72 patients × 500 genes (SIMULATED leukaemia data) ──
def em_gmm_diag(X, K, n_iter=300, var_floor=1e-3, tol=1e-6, rng=None, init_means=None):
    """EM for a GMM with diagonal covariances (genes independent within a cluster), with a variance floor.
    Starts from init_means, or from K randomly chosen samples."""
    rng = rng if rng is not None else np.random.default_rng(0)
    N, J = X.shape
    mus = X[rng.choice(N, K, replace=False)].copy() if init_means is None else np.array(init_means, float)
    vars_, pis = np.tile(X.var(axis=0), (K, 1)), np.full(K, 1.0 / K)
    trace = []
    for t in range(n_iter):
        log_joint = np.log(pis) - 0.5 * (np.log(2 * np.pi * vars_).sum(axis=1)
                                         + (((X[:, None, :] - mus[None]) ** 2) / vars_[None]).sum(axis=2))
        log_px = logsumexp(log_joint, axis=1)
        trace.append(log_px.sum())
        gamma = np.exp(log_joint - log_px[:, None])                            # E-step
        if t > 0 and abs(trace[-1] - trace[-2]) < tol:
            break
        Nk = gamma.sum(axis=0)                                                 # M-step
        pis, mus = Nk / N, gamma.T @ X / Nk[:, None]
        vars_ = np.maximum(gamma.T @ X ** 2 / Nk[:, None] - mus ** 2, var_floor)
    return dict(pis=pis, mus=mus, vars=vars_, gamma=gamma, log_joint=log_joint, trace=np.array(trace))


var_order = np.argsort(leuk["X"].var(axis=0))[::-1]
Xc = leuk["X"][:, var_order[:500]]
Xc = Xc - Xc.mean(axis=0)
y_leuk, subtype = leuk["y"], leuk["subtype"]
print("named marker genes among the 500 most variable:", sum(not g.startswith("G") for g in leuk["genes"][var_order[:500]]))
fits_d = [em_gmm_diag(Xc, 2, rng=np.random.default_rng(s)) for s in range(10)]
fit_d = max(fits_d, key=lambda f: f["trace"][-1])
lab_d = fit_d["gamma"].argmax(axis=1)
print("final log-likelihoods of 10 starts:", np.round(sorted(f["trace"][-1] for f in fits_d), 1))
print(f"ARI with ALL/AML: {adjusted_rand_score(y_leuk, lab_d):.2f}")
print(pd.crosstab(lab_d, subtype, rownames=["cluster"], colnames=["true subtype"]))
skd = GaussianMixture(2, covariance_type="diag", reg_covar=1e-3, n_init=10, random_state=SEED).fit(Xc)
print(f"scikit-learn diag GMM: ARI {adjusted_rand_score(y_leuk, skd.predict(Xc)):.2f}, "
      f"log-likelihood {skd.score(Xc) * len(Xc):.1f} (ours {fit_d['trace'][-1]:.1f})")
log_odds = fit_d["log_joint"][:, 0] - fit_d["log_joint"][:, 1]
print(f"patients with max responsibility < 0.99: {np.sum(fit_d['gamma'].max(1) < 0.99)} of 72; "
      f"smallest |log-odds| between the clusters: {np.abs(log_odds).min():.1f}")

🔍 **What just happened?** The diagonal GMM with two components rediscovers the ALL/AML split perfectly (ARI 1.00): one
cluster holds all 25 AML patients, the other all 47 ALL patients, B- and T-lineage together. Different random starts
again end at different log-likelihoods (local optima); the best equals scikit-learn's best of ten. Notice that the
500 most variable genes include most of the named marker genes — the genes that separate the classes are also the genes
that vary most, which is why unsupervised clustering can find the classes at all.

**Class discovery: is there a third group?** With $K = 3$ we can ask whether the mixture also separates the two ALL
lineages. Section 7.1 taught us to use many starts and a k-means initialisation — let us see why on this data set.

In [ ]:
# ── 10.2 K = 3: random starts vs a k-means start, and a ground-truth check against the true subtypes ──
fits3 = [em_gmm_diag(Xc, 3, rng=np.random.default_rng(s)) for s in range(30)]
ll3 = np.array([f["trace"][-1] for f in fits3])
_, cen3 = kmeans_hard(Xc, 3, n_starts=30)                                  # k-means needs restarts too
fit3 = em_gmm_diag(Xc, 3, init_means=cen3)                                  # k-means initialisation
lab3 = fit3["gamma"].argmax(axis=1)
print(f"30 random starts: best ℓ = {ll3.max():.1f}, reached (±1) by {np.sum(ll3 > ll3.max() - 1)} of 30; "
      f"median ℓ = {np.median(ll3):.1f}")
print(f"k-means-initialised EM: ℓ = {fit3['trace'][-1]:.1f}")
print(pd.crosstab(lab3, subtype, rownames=["cluster"], colnames=["true subtype"]))
print(f"🧪 ARI with the true subtypes (B-ALL / T-ALL / AML): {adjusted_rand_score(subtype, lab3):.2f}")


def bic_diag(f, X):
    """BIC = −2ℓ + p log N for a diagonal GMM: (K − 1) weights + K·J means + K·J variances."""
    K, J = f["mus"].shape
    return -2 * f["trace"][-1] + ((K - 1) + 2 * K * J) * np.log(len(X))


fit1 = em_gmm_diag(Xc, 1)
print("BIC: " + ", ".join(f"K = {K}: {bic_diag(f, Xc):,.0f}" for K, f in [(1, fit1), (2, fit_d), (3, fit3)]))

🔍 **What just happened?** With $K = 3$, most random starts end in poor local optima (the median run is about a
thousand log-likelihood units below the best); only two of the thirty find the best solution. The k-means start
reaches it in one run — and that solution is exactly the biology: 🧪 the three clusters are B-ALL, T-ALL and AML, with
ARI 1.00 against the true subtypes. This is class discovery of the kind Golub *et al.* hoped for, and with simulated
data we can verify it. (k-means has local optima of its own: with only 10 starts instead of 30 it splits the B-ALL
patients instead of finding T-ALL here — try `n_starts=10`. Restarts are needed at every level.)

Yet **BIC prefers $K = 2$**. Each extra component of a diagonal GMM in 500 dimensions costs 1,001 parameters, and with
only 9 T-ALL patients the gain in log-likelihood does not pay BIC's penalty of $1{,}001\times\log 72\approx 4{,}300$.
A criterion answers the question it was built for — "which model predicts new data best, given so few patients?" —
not "how many biological groups exist?". A small, real subgroup can be invisible to BIC and visible to a biologist.

### How confident is the model?

In [ ]:
# ── 10.3 Responsibilities and posterior odds of the K = 2 model; the weakest AML patients ──
weak = np.flatnonzero(leuk["atypical"])                                    # the simulator's weak-signature AML patients
score = (leuk["X"] - leuk["X"].mean(axis=0)) @ leuk["true_effect"]       # strength of the true AML signature
print("true-signature score: lowest AML patients", np.round(np.sort(score[y_leuk == 1])[:4], 1), "… | ALL maximum",
      np.round(score[y_leuk == 0].max(), 1), "| atypical patients", np.round(score[weak], 1))
print("🧪 the two patients with the smallest |log-odds|: sample_id",
      np.sort(leuk["sample_id"][np.argsort(np.abs(log_odds))[:2]]),
      "| the simulator's atypical AML patients: sample_id", leuk["sample_id"][weak])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
p_aml_cluster = fit_d["gamma"][:, np.bincount(lab_d[y_leuk == 1], minlength=2).argmax()]
for cls, col, name in [(0, CLASS_COLORS[0], "ALL"), (1, CLASS_COLORS[1], "AML")]:
    axes[0].hist(p_aml_cluster[y_leuk == cls], bins=np.linspace(0, 1, 21), color=col, alpha=0.7, label=name)
axes[0].set_xlabel("responsibility of the 'AML-like' cluster"); axes[0].set_ylabel("patients"); axes[0].legend()
axes[0].set_title("SIMULATED leukaemia data: soft assignments that are not soft", fontsize=9)
axes[1].hist(np.log10(np.abs(log_odds)), bins=20, color="0.5")
for k, n in enumerate(weak):
    axes[1].axvline(np.log10(abs(log_odds[n])), color="tab:orange", lw=1.5, label="two weakest AML patients" if k == 0 else None)
axes[1].axvline(np.log10(np.log(99)), color="tab:red", ls="--", label="99 % : 1 %")
axes[1].set_xlabel("log10 |log posterior odds| between the two clusters"); axes[1].set_ylabel("patients")
axes[1].set_title("posterior odds are astronomically large", fontsize=9); axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Look at the responsibilities: **every** patient is assigned with probability essentially 0 or
1. The smallest absolute posterior log-odds is about 76 — odds of roughly $e^{76}\approx10^{33}$ to 1 — and most are in
the hundreds, far beyond the 99 %:1 % line (log-odds 4.6). The simulation lets us look closer: two AML patients
(sample_id 47 and 67, flagged by the simulator's `atypical` mask) were generated with a deliberately **weak** AML
signature, and they are exactly the two patients with the smallest log-odds (orange lines). The model ranks them correctly as the least typical — but still declares them AML with
odds of $10^{33}$ and more, which no haematologist would accept for borderline cases.

Why? The E-step adds up 500 per-gene log-likelihood ratios as if the genes were **independent** evidence. In reality,
co-expressed genes (Module 12's gene blocks; the simulated data contain 15 hidden co-expression programmes) carry largely
the *same* information, so the model counts it many times — the same over-confidence as naive Bayes.
**Responsibilities are probabilities under the model, and only as trustworthy as the model's assumptions.** Modelling the
dependence between genes — for example with a few principal components ([Module 14](14_pca_dimensionality_reduction.ipynb))
and full covariances — is one route to more honest uncertainty; try it once you have met PCA.

✏️ **Your turn.** Repeat 10.1 with only the 10 most variable genes (`var_order[:10]`; print their names with
`leuk["genes"][var_order[:10]]`). Which split does the two-cluster model find now, and why? How many patients have a
responsibility below 0.99?

---
## 11 · Comparing and evaluating clusterings

> 📍 **Where we are.** We now have three clustering methods (Module 12's hierarchical clustering and k-means, and this
> module's mixtures). The table below compares them; then comes the hardest question — what is a *good* clustering?

| | Hierarchical | k-means | GMM |
|---|---|---|---|
| **Running time** | naively $O(N^3)$ | fastest (each iteration is linear in $N$) | fast (each iteration is linear in $N$) |
| **Assumptions** | requires a similarity / distance measure | strong assumptions | strongest assumptions |
| **Input parameters** | none | $K$ | $K$ |
| **Clusters** | subjective (only a tree is returned) | exactly $K$ clusters | exactly $K$ clusters |

Two comments. (i) "Linear" refers to $N$: a full-covariance GMM iteration costs about $O(NKJ^2+KJ^3)$, which is very
slow for thousands of genes — one more reason for diagonal covariances. (ii) "Strongest assumptions" means the GMM
states its assumptions *explicitly* as a probability model; in terms of cluster **shapes** a full-covariance GMM is the
most flexible of the three (k-means is its special case with equal round clusters, Section 6). In exchange for the
explicit model you get probabilities, a likelihood, BIC and a principled way to handle uncertainty.

**What is a good clustering?**

- **Internal criteria** use only the data: high **intra**-cluster similarity, low **inter**-cluster similarity. A popular
  score is the **silhouette** (Rousseeuw 1987). For sample $n$ let $a(n)$ be its mean distance to the other members of its
  own cluster and $b(n)$ its mean distance to the members of the **nearest other** cluster:
  $$s(n)=\frac{b(n)-a(n)}{\max\{a(n),b(n)\}}\in[-1,1].$$
  *In words:* $s \approx 1$ — snugly inside its cluster; $s\approx0$ — on a border; $s<0$ — probably in the wrong cluster.
  The average over all samples summarises a clustering.
- **External criteria** compare with outside knowledge: known labels (via the **adjusted Rand index**), or — for genes —
  annotations such as the **Gene Ontology** (GO): are the genes in a cluster enriched for a biological process?

In [ ]:
# ── 11.1 The silhouette, from scratch, for the GMM fits of Section 8 ──
def silhouette_values(D, labels):
    """Silhouette s(n) = (b − a) / max(a, b) for every sample, from a full distance matrix D."""
    s = np.zeros(len(labels))
    for n in range(len(labels)):
        own = (labels == labels[n]); own[n] = False
        if not own.any():
            continue                                         # a singleton cluster gets s = 0 by convention
        a = D[n, own].mean()
        b = min(D[n, labels == k].mean() for k in np.unique(labels) if k != labels[n])
        s[n] = (b - a) / max(a, b)
    return s


D_train = np.sqrt(((X_train[:, None, :] - X_train[None]) ** 2).sum(axis=2))
rows = []
for K in range(2, 8):
    lab = fits_by_K[K]["gamma"].argmax(axis=1)
    rows.append({"K": K, "mean silhouette (ours)": silhouette_values(D_train, lab).mean(),
                 "scikit-learn": silhouette_score(X_train, lab), "BIC": res.loc[K, "BIC"]})
print(pd.DataFrame(rows).set_index("K").round(3))

🔍 **What just happened?** Our silhouette matches scikit-learn's. But the two criteria **disagree**: the mean silhouette
is (marginally) highest for $K = 2$ (0.497 vs 0.494 for $K = 3$), while BIC prefers $K = 3$. The silhouette rewards
compact, well-separated groups in Euclidean distance, and our tall and diagonal clusters overlap enough to look like one
blob by that standard; BIC rewards whatever a Gaussian mixture can describe. Two criteria, two notions of "good" — report
which one you used, and why.

### The adjusted Rand index (external)

Module 12 used scikit-learn's ARI; here is where it comes from. Take all $\binom{N}{2}$ **pairs** of samples. A pair is
handled consistently if both partitions put it together, or both put it apart; the **Rand index** is the fraction of
consistent pairs. The **adjusted** version (Hubert & Arabie 1985) subtracts the value expected by chance. From the
contingency table $n_{kc}$ (samples in cluster $k$ and class $c$) with row sums $a_k$ and column sums $b_c$:

$$\mathrm{ARI}=\frac{\sum_{kc}\binom{n_{kc}}{2}-E}{\tfrac12\big[\sum_k\binom{a_k}{2}+\sum_c\binom{b_c}{2}\big]-E},
\qquad E=\frac{\sum_k\binom{a_k}{2}\sum_c\binom{b_c}{2}}{\binom{N}{2}} .$$

*In words:* count the pairs that are together in both partitions, subtract what chance would give, and rescale so that
identical partitions score 1 and random ones about 0. Renaming clusters does not change it.

In [ ]:
# ── 11.2 ARI from scratch ──
def adjusted_rand(y_true, y_pred) -> float:
    """Adjusted Rand index from the contingency table (Hubert & Arabie 1985)."""
    _, t = np.unique(y_true, return_inverse=True)
    _, p = np.unique(y_pred, return_inverse=True)
    ct = np.zeros((p.max() + 1, t.max() + 1))
    np.add.at(ct, (p, t), 1)

    def pairs(v):
        return v * (v - 1) / 2                                  # number of pairs among v objects

    together_both = pairs(ct).sum()
    sum_a, sum_b = pairs(ct.sum(axis=1)).sum(), pairs(ct.sum(axis=0)).sum()
    expected = sum_a * sum_b / pairs(len(y_true))
    return float((together_both - expected) / (0.5 * (sum_a + sum_b) - expected))


rng = np.random.default_rng(SEED + 112)
tests = {"GMM (K = 3) vs truth": (c_train, fits_by_K[3]["gamma"].argmax(1)),
         "GMM (K = 6) vs truth": (c_train, fits_by_K[6]["gamma"].argmax(1)),
         "k-means (K = 3) vs truth": (c_train, kmeans_hard(X_train, 3)[0]),
         "truth with renamed labels": (c_train, (c_train + 1) % 3),
         "random labels": (c_train, rng.integers(0, 3, len(c_train)))}
for name, (a_, b_) in tests.items():
    print(f"{name:28s} ARI ours {adjusted_rand(a_, b_):6.3f} | scikit-learn {adjusted_rand_score(a_, b_):6.3f}")

🔍 **What just happened?** Our formula reproduces scikit-learn for every case. Renaming the labels leaves ARI at exactly
1; random labels give about 0; the GMM with the right $K$ (0.90) scores higher than k-means (0.76), whose round-cluster
assumption mis-assigns samples of the elongated, overlapping groups, and much higher than the over-split $K = 6$ fit (0.61).

### GO enrichment as an external criterion

Suppose a cluster of co-expressed yeast genes contains genes annotated with the GO term "DNA replication". Is that more
than chance? The classic test is **hypergeometric** — drawing balls from an urn without replacement. With $M$ genes in
the genome, $n$ of them annotated with the term, and a cluster of $N_c$ genes of which $k$ are annotated:

$$P(X\ge k)=\sum_{x=k}^{\min(n,N_c)}\frac{\binom{n}{x}\binom{M-n}{N_c-x}}{\binom{M}{N_c}} .$$

*In words:* if the cluster were a random handful of $N_c$ genes, how often would it contain at least $k$ annotated ones?
A tiny probability means the cluster is **enriched** for the process. The numbers below are invented toy values
(≈ 6,000 genes is the size of the yeast genome).

In [ ]:
# ── 11.3 GO enrichment of one gene cluster: hypergeometric test (toy numbers) ──
M, n_annot, N_c, k_obs = 6000, 90, 50, 8          # genome, genes with the GO term, cluster size, annotated in cluster
p_hand = sum(math.comb(n_annot, x) * math.comb(M - n_annot, N_c - x) for x in range(k_obs, min(n_annot, N_c) + 1)) \
         / math.comb(M, N_c)
p_scipy = stats.hypergeom(M, n_annot, N_c).sf(k_obs - 1)          # P(X ≥ k) = 1 − P(X ≤ k − 1)
expected = N_c * n_annot / M
print(f"expected annotated genes in a random cluster of {N_c}: {expected:.2f}; observed {k_obs} "
      f"(fold enrichment {k_obs / expected:.1f})")
print(f"P(X ≥ {k_obs}) by hand = {p_hand:.2e}; SciPy = {p_scipy:.2e}")
print(f"testing ~5,000 GO terms: Bonferroni threshold 0.05 / 5000 = {0.05 / 5000:.0e} → "
      f"{'still significant' if p_hand < 0.05 / 5000 else 'not significant'}")

🔍 **What just happened?** A random cluster of 50 genes would contain fewer than one "DNA replication" gene on average;
8 is more than ten times that, and the hypergeometric probability of so many by chance is tiny — far below the
Bonferroni threshold needed when thousands of GO terms are tested at once. Our hand-written sum equals SciPy's.

⚠️ Enrichment supports a biological interpretation of a cluster, but it is not proof that the clustering is "right":
annotations are incomplete and biased towards well-studied genes, and many clusterings of the same data can show some
enrichment. External criteria, like internal ones, are evidence to weigh — not a final verdict.

✏️ **Your turn.** Set `k_obs = 3` in cell 11.3. Is 3 annotated genes (four times the expectation) significant on its
own? After the Bonferroni correction for 5,000 GO terms?

### 📘 Lecture notes — comparing and evaluating clusterings

**Comparison of clustering methods.**

| | Hierarchical | k-means | GMM |
|---|---|---|---|
| running time | naively $O(N^3)$ | fastest (each iteration linear in $N$) | fast (each iteration linear in $N$) |
| assumptions | a similarity / distance measure | strong | strongest (an explicit probability model) |
| input parameters | none | $K$ | $K$ |
| output | a tree — clusters are a subjective cut | exactly $K$ clusters (hard) | exactly $K$ clusters (soft) |

**What is a good clustering?**
- *Internal criterion:* high intra-cluster similarity and low inter-cluster similarity, e.g. the silhouette
  $s(n)=\big(b(n)-a(n)\big)/\max\{a(n),b(n)\}\in[-1,1]$ ($a$: mean distance to its own cluster, $b$: to the nearest
  other cluster).
- *External criteria:* compare with outside resources — known labels (adjusted Rand index: 1 = identical partitions,
  ≈ 0 = chance) or, for gene clusters, **Gene Ontology enrichment**: do the genes of a cluster overlap with genes
  annotated with the same biological process more than chance allows (hypergeometric test, corrected for testing
  thousands of terms)?

**Summary of the lecture.** Mixture models are the probabilistic version of k-means; the EM algorithm learns them.

> 🎓 **Exam tip.** Know one internal criterion (silhouette: definition and range) and one external criterion (ARI or GO enrichment),
    and be able to fill in the comparison table from memory with one reason per cell.

## 📌 Key takeaways

- **Soft clustering**: a mixture model assigns each sample a probability for every cluster (rows sum to 1); k-means'
  hard 0/1 table hides the uncertainty of samples between clusters.
- **The model** (the course's two-step template): $c\sim\mathrm{Categorical}(\pi)$, $\mathbf x\mid c=i\sim\mathcal N(\boldsymbol\mu_i,\Sigma_i)$,
  so $p(\mathbf x)=\sum_i\pi_i\mathcal N(\mathbf x\mid\boldsymbol\mu_i,\Sigma_i)$ — a two-node Bayesian network $c\to x$ and the
  building block of HMMs.
- **Inference** is Bayes' rule: $\gamma_{ni}\propto\pi_i\,\mathcal N(\mathbf x^n\mid\boldsymbol\mu_i,\Sigma_i)$, computed in log space.
- **Learning with labels** is counting and averaging; **with hidden labels** it is **EM**: E-step = responsibilities,
  M-step = responsibility-weighted MLEs — *latent variables ⇒ EM*, the course's third recurring idea.
- **EM's guarantee, precisely:** the observed-data log-likelihood never decreases and (if bounded) converges to a
  stationary point, usually a local maximum — **not** necessarily the global one; the unconstrained GMM likelihood is
  unbounded (degenerate solutions), so use restarts, k-means initialisation and variance floors.
- **k-means** is EM for a GMM with equal weights and shared spherical covariance $\sigma^2I$ in the limit $\sigma\to0$.
- **Choosing K**: not by training likelihood, but by held-out likelihood, BIC, or a Dirichlet-process prior (Chinese
  restaurant process) — and a criterion can miss a small, real subgroup (T-ALL in Section 10).
- **Other mixtures** — Bernoulli/binomial mixtures recover **population structure** from genotypes; compare methods by
  running time, assumptions, inputs and output; evaluate clusterings with internal (silhouette) and external (ARI, GO
  enrichment) criteria, and remember that model-based probabilities are only as honest as the model's assumptions.

---
## ✅ Checkpoint

1. A two-cluster 1-D GMM has π = (0.8, 0.2). For a sample x, N(x | μ₁, σ₁²) = 0.05 and N(x | μ₂, σ₂²) = 0.40. Compute p(c = 2 | x).
   <details><summary>Answer</summary>Joint terms: 0.8 × 0.05 = 0.04 and 0.2 × 0.40 = 0.08. p(x) = 0.12, so p(c = 2 | x) = 0.08/0.12 ≈ 0.67 — the larger likelihood beats the smaller prior.</details>
2. How many free parameters does a GMM with K = 4 components and full covariances have in J = 3 dimensions? With diagonal covariances?
   <details><summary>Answer</summary>Full: (K − 1) + KJ + K·J(J+1)/2 = 3 + 12 + 4·6 = 39. Diagonal: 3 + 12 + 4·3 = 27.</details>
3. Write the M-step update for μᵢ and explain in words how it differs from the labelled-data MLE.
   <details><summary>Answer</summary>$\mu_i=\sum_n\gamma_{ni}x^n/\sum_n\gamma_{ni}$. It is the labelled-data mean with the 0/1 indicator $I(c^n=i)$ replaced by the responsibility $\gamma_{ni}$: every sample contributes fractionally to every cluster.</details>
4. True or false: "EM always finds the maximum-likelihood estimate of a Gaussian mixture." Explain.
   <details><summary>Answer</summary>False. EM never decreases the observed-data log-likelihood and converges (when the likelihood is bounded) to a stationary point, usually a local maximum, which depends on the initialisation. Moreover, the unconstrained GMM likelihood is unbounded (a component collapsing on one point gives ℓ → ∞), so a global MLE does not even exist.</details>
5. In our leukaemia example (Section 7.2), unconstrained EM produced σ₁ → 0. What happened, and name two remedies.
   <details><summary>Answer</summary>One component collapsed onto the many identical values at the preprocessing floor (log10 100 = 2), so its density there — and the likelihood — grew without bound. Remedies: a variance floor / regulariser (reg_covar), a prior on the variances (MAP), or modelling the floored values as censored.</details>
6. Under which assumptions does EM for a GMM become k-means?
   <details><summary>Answer</summary>Equal mixing proportions, a shared spherical covariance σ²I that is fixed (not learned), and the limit σ → 0: the responsibilities become 0/1 indicators of the nearest centre and the M-step for the means becomes the cluster mean.</details>
7. Training log-likelihoods for K = 1…5 are −900, −700, −610, −600, −595 with N = 200 samples in 2-D (full covariance). Which K does BIC choose? (log 200 ≈ 5.3)
   <details><summary>Answer</summary>p(K) = (K−1) + 2K + 3K = 6K − 1 → 5, 11, 17, 23, 29. BIC = −2ℓ + 5.3p: K=1: 1800 + 26.5 = 1826.5; K=2: 1400 + 58.3 = 1458.3; K=3: 1220 + 90.1 = 1310.1; K=4: 1200 + 121.9 = 1321.9; K=5: 1190 + 153.7 = 1343.7. BIC chooses K = 3.</details>
8. In `em_binomial_mixture`, why is the M-step denominator `2 * Nk + 2 * a`, and what would go wrong with a = 0?
   <details><summary>Answer</summary>Each individual carries two alleles per SNP, so the weighted number of chromosomes in cluster i is 2Σₙγₙᵢ; the pseudo-counts add a allele copies of each kind (2a chromosomes). With a = 0 an allele never seen in a cluster gives μ = 0 or 1 and log 0 = −∞ in the next E-step — a degenerate solution.</details>

## 🧠 Questions & discussion

1. A single-cell RNA-seq study clusters cells with a GMM and reports that 8 % of cells have no responsibility above 0.9. Are these cells in transition between states, doublets, poor-quality cells — or artefacts of a wrong model? How would you find out?
2. Section 10 showed that a diagonal GMM on 500 genes is wildly over-confident. Which assumption is responsible, and how would you change the model or the features to get calibrated uncertainty?
3. BIC and held-out likelihood both chose K = 3 for simulated Gaussian clusters. If the true clusters were banana-shaped, what would BIC do, and is that a problem for biological interpretation?
4. STRUCTURE gives admixture proportions, our binomial mixture gives cluster responsibilities. For a person with one European and one East Asian parent, what would each method report?
5. The EM guarantee is about the likelihood, not about biology. Give an example where a higher likelihood corresponds to a *less* meaningful clustering (hint: Section 7.2).

## 🚀 Bonus — going further

- **Mixtures of other distributions.** Replace the Gaussian by a Poisson to cluster count data (e.g. reads per gene): derive the M-step (λᵢ = weighted mean count) and implement it on simulated counts.
- **Gibbs sampling for mixtures.** Instead of soft responsibilities, *sample* each label from its posterior, then sample the parameters — the sampling sibling of EM ([Module 21](21_bayesian_learning_mcmc.ipynb)). Compare its results with EM on the three-cluster data.
- **Baum–Welch preview.** In an HMM the E-step needs the posterior of every hidden state given the *whole* sequence (forward–backward). Read [Module 20](20_hmm_learning.ipynb) and identify which formulas of this notebook reappear.
- **Cryo-ET classification.** The sister course's Module 04 runs EM for GMMs, BIC, k-means, hierarchical clustering, the silhouette and the ARI on cryo-electron-tomography particles, and links them to the lab's FAML and DISCA papers: [`../../../../cryoem/tutorial/BiologyToResearch/`](../../../../cryoem/tutorial/BiologyToResearch/).
- **From EM to variational inference.** EM's lower bound (Section 5.3) is the starting point of variational inference and variational autoencoders, which learn mixtures and latent spaces with neural networks — Level 2, [Module 27](27_variational_inference_vae.ipynb).
- **Probability foundations.** For more on likelihood, MLE/MAP and uncertainty, see [`../../FundamentalsLearningUncertainty/`](../../FundamentalsLearningUncertainty/); for generative classifiers built on the same class-conditional Gaussians, [`../../GenerativeDiscriminativeClassification/`](../../GenerativeDiscriminativeClassification/).

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 13 — Mixture models and the EM algorithm        Date:
Input: SIMULATED gene time courses and genotypes (Section 1), 1-D marker gene (400 cells), 2-D three-cluster and
       five-group data, genotypes of 3 populations (300 × 500 SNPs, F_ST = 0.05); SIMULATED leukaemia data
       72 × 3,571 (shaped like Golub et al. 1999)
Output: responsibilities, EM fits and log-likelihood traces, BIC / held-out curves, population clusters, ARI, silhouette
Method: Bayes' rule inference; EM for 1-D, full-covariance, diagonal and binomial mixtures; restarts; variance floors
Key equation: γ_ni = π_i N(x_n | μ_i, Σ_i) / Σ_j π_j N(x_n | μ_j, Σ_j);  M-step = γ-weighted MLEs;  ℓ(θ^(t+1)) ≥ ℓ(θ^(t))
Key parameters: K; covariance type; initialisation and number of restarts; variance floor / pseudo-count; tol
What changed when I modified …: (e.g. σ in the k-means limit; F_ST and #SNPs; the variance floor)
One failure mode: (e.g. a component collapsing on the detection floor; local optimum with 5 groups or K = 3 subtypes)
Ground-truth check: (e.g. K = 3 clusters vs the true B-ALL / T-ALL / AML subtypes)
Connection to the 02-620 topics (which idea of the lecture this illustrates):
       mixture models (soft clustering, GMM, inference, MLE, EM, k-means limit, EM in practice, number of
       clusters, Bernoulli mixtures, evaluating clusterings)
```

## 📚 References

- Carnegie Mellon University 02-620 *Machine Learning for Scientists* (course design S. Kim; instructors M. Xu and M. Zhang) — this module teaches the material of its lecture on mixture models and the EM algorithm, with the comparison to the clustering methods of the preceding lecture.
- Murphy K. P. (2012) *Machine Learning: A Probabilistic Perspective*, chapter 11 (mixture models and the EM algorithm). MIT Press.
- Hastie T., Tibshirani R., Friedman J. (2009) *The Elements of Statistical Learning*, 2nd ed., §8.5 (the EM algorithm) and §14.3.7 (Gaussian mixtures as soft k-means). Springer.
- Dempster A. P., Laird N. M., Rubin D. B. (1977) Maximum likelihood from incomplete data via the EM algorithm. *JRSS-B* 39:1–38.
- Wu C. F. J. (1983) On the convergence properties of the EM algorithm. *Annals of Statistics* 11:95–103.
- Schwarz G. (1978) Estimating the dimension of a model. *Annals of Statistics* 6:461–464.
- Ferguson T. S. (1973) A Bayesian analysis of some nonparametric problems. *Annals of Statistics* 1:209–230.
- Pritchard J. K., Stephens M., Donnelly P. (2000) Inference of population structure using multilocus genotype data. *Genetics* 155:945–959.
- Alexander D. H., Novembre J., Lange K. (2009) Fast model-based estimation of ancestry in unrelated individuals. *Genome Research* 19:1655–1664.
- Balding D. J., Nichols R. A. (1995) A method for quantifying differentiation between populations at multi-allelic loci and its implications for investigating identity and paternity. *Genetica* 96:3–12.
- Golub T. R. *et al.* (1999) Molecular classification of cancer: class discovery and class prediction by gene expression monitoring. *Science* 286:531–537 (the study the simulated leukaemia data imitate).
- Dudoit S., Fridlyand J., Speed T. P. (2002) Comparison of discrimination methods for the classification of tumors using gene expression data. *JASA* 97:77–87 (the preprocessing, including the detection floor, that the simulated data imitate).
- Rousseeuw P. J. (1987) Silhouettes: a graphical aid to the interpretation and validation of cluster analysis. *J Comput Appl Math* 20:53–65.
- Hubert L., Arabie P. (1985) Comparing partitions. *Journal of Classification* 2:193–218.
- Ashburner M. *et al.* (2000) Gene Ontology: tool for the unification of biology. *Nature Genetics* 25:25–29.

## ➡️ Next

Mixture models found structure by grouping samples. [Module 14 · Principal component analysis](14_pca_dimensionality_reduction.ipynb)
finds structure by **compressing** variables: it projects thousands of genes or SNPs onto a few axes of largest
variance — and, applied to European genomes, draws a map of Europe.